# Sesión 3 — MUBD

## Modelo lineal: de los datos a la predicción

En la sesión 2 estimamos y contrastamos medias, varianzas y proporciones. Ahora queremos **predecir una variable numérica a partir de otras variables**: ¿cómo se relaciona la resistencia del hormigón (`Strength`) con su composición y su edad?

Seguiremos el recorrido **describir → ajustar → seleccionar → diagnosticar → transformar → interpretar → evaluar en test**. Un buen ajuste en entrenamiento no garantiza buenas predicciones con observaciones nuevas.

**Cómo trabajar:** lee la explicación, ejecuta el código comentado, interpreta la salida y resuelve la micropráctica. Los huecos `...` se completan con funciones y patrones ya mostrados; los ejercicios 3.1 y 3.2 permiten practicar apartados más amplios.

Selecciona el kernel del entorno del proyecto y ejecuta las celdas **en orden** con **Shift + Enter**. Si falta una variable, ejecuta las celdas anteriores. Para repetir el análisis desde cero, reinicia el kernel y ejecuta todo.

**Regla del análisis:** conserva `datos` y los modelos originales. Ajustaremos y seleccionaremos solo con entrenamiento; el test se reserva para evaluar el modelo final, no para elegirlo.

## 3.0. Preparación y lectura de datos

### Bibliotecas y entorno

Usamos **pandas** para leer y resumir tablas, **NumPy** para el cálculo numérico y **Matplotlib** para los gráficos. Más adelante, **statsmodels** ajustará los modelos y **SciPy** aportará contrastes y transformaciones.

La celda también importa herramientas que se reutilizarán después y fija el formato de tablas y figuras. Los algoritmos largos permanecen en módulos auxiliares junto al cuaderno; explicaremos su propósito cuando los utilicemos.

**Antes de ejecutar:** prepara las dependencias del proyecto y selecciona el kernel de ese mismo entorno. No es necesario instalar bibliotecas en cada ejecución del cuaderno.

In [ ]:
from pathlib import Path  # Construye rutas a las carpetas y archivos.
import warnings  # Permite controlar avisos numéricos durante los suavizados.

import numpy as np  # Aporta operaciones numéricas y generación de números aleatorios.
import pandas as pd  # Permite leer, seleccionar y resumir tablas de datos.
import matplotlib.pyplot as plt  # Crea figuras y gráficos.
import statsmodels.api as sm  # Ajusta modelos estadísticos por mínimos cuadrados.
from IPython.display import display  # Muestra tablas y resultados con formato.
from pandas.plotting import scatter_matrix  # Dibuja una matriz de gráficos de dispersión.
from scipy import stats  # Aporta distribuciones, contrastes y transformaciones.
from scipy.optimize import minimize_scalar  # Busca el óptimo de una función de una variable.
from statsmodels.nonparametric.smoothers_lowess import lowess  # Resume tendencias con un suavizado local.
from threadpoolctl import threadpool_limits  # Limita los hilos en cálculos posteriores de influencia.

# Muestra los gráficos dentro del cuaderno.
%matplotlib inline
# Muestra todas las expresiones de salida de una celda, no solo la última.
%config InteractiveShell.ast_node_interactivity = "all"
plt.rcParams.update({"figure.figsize": (16, 8), "figure.dpi": 100,  # Fija el tamaño y la resolución iniciales.
                     "axes.titlesize": 20, "axes.labelsize": 16,  # Ajusta títulos y etiquetas de los ejes.
                     "xtick.labelsize": 14, "ytick.labelsize": 14})  # Ajusta el tamaño de las marcas de los ejes.
pd.set_option("display.max_columns", 20)  # Permite mostrar las nueve variables sin ocultar columnas.

### 3.0.1. Lectura del conjunto de entrenamiento

Abrimos la tabla de hormigón que utilizaremos para ajustar los modelos. El archivo está separado por **tabulaciones** (`sep="\t"`) y su primera fila contiene los nombres de las variables. `NA` y las celdas vacías se reconocen como valores ausentes.

La ruta `Path.cwd().parent.parent / "datos"` supone que el directorio de trabajo sea el de este cuaderno, dentro de `sesiones/sesion3`: sube dos niveles y entra en la carpeta de datos. Si aparece un error de archivo no encontrado, comprueba el directorio de trabajo del kernel.

**Qué queremos predecir:**

- **Respuesta:** `Strength`, resistencia a compresión del hormigón, en MPa.
- **Predictoras de composición:** `Cement` (cemento), `BlastFurnaceSlag` (escoria), `FlyAsh` (ceniza volante), `Water` (agua), `Superplasticizer` (superplastificante), `CoarseAggregate` (árido grueso) y `FineAggregate` (árido fino).
- **Predictora de edad:** `Age`, edad del hormigón en días.

Numeramos las filas de 1 a $n$ y llamamos al índice `observacion`. **Son etiquetas para localizar casos, no fechas ni un orden temporal demostrado.** Guardamos los nombres originales para no confundirlos después con las variables transformadas.

In [ ]:
DATA_PATH = Path.cwd().parent.parent / "datos"  # Localiza los datos desde el directorio de este cuaderno.

datos = pd.read_csv(DATA_PATH / "Concrete_train.txt", sep="\t", na_values=["NA", ""])  # Lee el entrenamiento tabulado y reconoce ausentes.
datos.index = np.arange(1, len(datos) + 1)  # Asigna etiquetas consecutivas de 1 al número de observaciones.
datos.index.name = "observacion"  # Da un nombre al índice que usaremos para localizar casos.
columnas_originales = list(datos.columns)  # Guarda los nombres antes de crear transformaciones.
predictores = [columna for columna in datos if columna != "Strength"]  # Selecciona las ocho variables de entrada, sin la respuesta.

## 3.1. Inspección inicial

Antes de ajustar un modelo, comprobamos **cuántas observaciones hay**, **qué contiene cada fila** y **cómo se distribuye cada variable**. Un archivo que se lee sin errores todavía puede tener columnas mal interpretadas o valores que debamos revisar.

### 3.1.1. Tamaño de la tabla

`datos.shape` devuelve el par **(filas, columnas)**. Las filas son observaciones de hormigón; las columnas incluyen tanto las predictoras como la respuesta. El índice `observacion` no cuenta como una columna de datos.

In [ ]:
datos.shape  # Devuelve el número de observaciones y de variables, en ese orden.

### Qué nos dice el tamaño de la tabla

La salida **(706, 9)** indica que disponemos de **706 observaciones** y **9 variables**: ocho predictoras y una respuesta, `Strength`. No son nueve predictoras; tampoco se incluye aquí el índice de las filas.

**Implicación para el análisis:** estas 706 filas forman el entrenamiento. El tamaño es importante para valorar la información disponible, pero no garantiza por sí solo que la muestra sea representativa ni que las observaciones sean independientes.

### 3.1.2. Primeras filas y selección de un caso

`head()` muestra las primeras cinco filas para revisar nombres y valores. Después, `loc[[5]]` recupera la observación **etiquetada 5** y conserva el formato de tabla. En este caso la etiqueta coincide con la quinta fila porque acabamos de numerar el índice; esa coincidencia no es una propiedad general de `loc`.

In [ ]:
display(datos.head())  # Muestra las primeras cinco observaciones con todas sus variables.
display(datos.loc[[5]])  # Recupera como tabla la observación cuya etiqueta es 5.

### Lectura de las primeras filas

La observación **5** tiene una edad de **90 días** y una resistencia observada de **38,07 MPa**. Ese valor es una medición de la tabla, **no una predicción**: todavía no hemos ajustado ningún modelo.

En las primeras filas aparecen ceros en `FlyAsh` y `Superplasticizer`. Un cero es un valor registrado y puede representar que no se empleó ese componente; **no equivale a un dato ausente** ni debe reemplazarse automáticamente.

**Implicación:** las primeras filas permiten detectar detalles de lectura y entender qué representa un caso, pero no describen por sí solas las 706 observaciones. Para eso usamos el resumen completo y, después, los gráficos.

### 3.1.3. Resumen descriptivo

`describe()` resume cada columna numérica: `count` es el número de valores válidos, `mean` la media, `std` la desviación típica muestral, `min` y `max` los extremos, y `25%`, `50%` y `75%` los cuartiles. **`50%` es la mediana.**

Compara centro y dispersión **dentro de cada variable**; no interpretes diferencias de escala entre composición, edad y resistencia como si todas midieran lo mismo. La desviación típica describe la variabilidad de las observaciones: no es el error estándar de una media ni una medida de error de predicción.

In [ ]:
display(datos.describe())  # Resume tamaño válido, centro, dispersión, cuartiles y extremos de cada variable.

### Qué nos dice el resumen descriptivo

- **Tamaño válido:** las nueve columnas tienen **706 valores numéricos**; sus recuentos coinciden con el número de filas y no muestran ausentes en estas variables.
- **Centro de la respuesta:** la resistencia media es **35,21 MPa** y la mediana, **33,71 MPa**. La media es algo mayor, pero esos dos números no bastan para describir la forma completa de la distribución.
- **Dispersión y extremos:** la desviación típica de `Strength` es **16,43 MPa** y los valores van de **2,33 a 82,60 MPa**. El 50 % central de las resistencias se sitúa aproximadamente entre **23,43 y 44,61 MPa**.
- **Otras variables que conviene revisar:** `Age` tiene mediana **28 días** y máximo **365 días**. Los mínimos de `BlastFurnaceSlag`, `FlyAsh` y `Superplasticizer` son cero; más adelante tendremos que respetar ese dominio si exploramos logaritmos.

**Implicación para el análisis:** la resistencia presenta variabilidad que intentaremos explicar con las predictoras. Este resumen todavía no indica qué variables predicen mejor, si las relaciones son lineales o si los errores de un modelo serán normales. Lo siguiente será explorar gráficamente las distribuciones y las relaciones.

## 3.2. Exploración gráfica

### 3.2.1. Diagramas de caja de las variables originales

El **boxplot** resume la mediana (línea central), el 50 % central (caja) y los posibles valores atípicos (puntos). Los bigotes llegan hasta los valores más alejados que quedan dentro de $1{,}5$ rangos intercuartílicos desde cada extremo de la caja.

Representamos todas las variables **en sus escalas originales**. Una caja de un componente con cantidades grandes puede comprimir visualmente la de `Strength`; no compares alturas entre composición, días y MPa como si fueran la misma medida.

**Pregunta antes de ejecutar:** ¿qué variables podrían mostrar muchos ceros o valores alejados de su centro? Un punto fuera de los bigotes es una señal para revisar, no una razón automática para eliminarlo.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))  # Crea una figura y un eje común para todas las cajas.
_ = ax.boxplot([datos[columna].dropna() for columna in columnas_originales],  # Una caja por variable, sin ausentes.
               patch_artist=True,  # Permite colorear el interior de las cajas.
               boxprops={"facecolor": "#9fc3e4", "edgecolor": "#666666", "linewidth": 1.2},  # Relleno y borde.
               whiskerprops={"color": "#666666"}, capprops={"color": "#666666"},  # Estilo de los bigotes.
               medianprops={"color": "#800000", "linewidth": 1.8})  # Destaca cada mediana en granate.
_ = ax.set_xticks(np.arange(1, len(columnas_originales) + 1))  # Asigna una posición a cada variable.
_ = ax.set_xticklabels(columnas_originales, rotation=45, ha="right")  # Gira los nombres para poder leerlos.
_ = ax.set_title("Distribución de las variables originales")  # Identifica el contenido del gráfico.
ax.tick_params(labelsize=14)  # Ajusta el tamaño de las etiquetas.
fig.tight_layout()  # Evita recortes de títulos y nombres.
plt.show()  # Muestra las cajas dentro del cuaderno.

### Lectura de los diagramas de caja

`Age` presenta varios valores altos alejados de su caja, coherentes con su máximo de 365 días. En `FlyAsh`, la mediana está en cero; en `Water` y `Strength` también aparecen puntos fuera de los bigotes.

**Qué permite concluir:** estas observaciones merecen una revisión contextual, no una eliminación automática. Los áridos ocupan la parte superior del gráfico por su escala numérica; eso no significa que sean las variables más importantes para predecir la resistencia.

**Siguiente paso:** las cajas describen cada variable por separado. Para estudiar cómo se relacionan dos variables necesitamos gráficos de dispersión.

### 3.2.2. Matriz de dispersión

Cada panel fuera de la diagonal representa un par de variables: la **columna** determina el eje horizontal y la **fila**, el vertical. En la diagonal se muestran histogramas. El mismo par aparece dos veces, con los ejes intercambiados.

Busca tres tipos de patrones:

1. **Asociación con `Strength`:** ¿la nube sube, baja o no presenta una tendencia clara?
2. **Forma:** ¿parece una recta, una curva o varias agrupaciones?
3. **Relación entre predictoras:** ¿dos columnas aportan información parecida?

La transparencia ayuda a ver acumulaciones de puntos. Este gráfico es exploratorio: una asociación no demuestra causalidad ni mide el efecto de una variable manteniendo constantes las demás.

In [ ]:
axes = scatter_matrix(datos[columnas_originales], figsize=(18, 18), diagonal="hist",  # Dibuja todos los pares y los histogramas.
                      color="#9fc3e4", alpha=0.5, s=16,  # Usa puntos pequeños y semitransparentes.
                      hist_kwds={"color": "#9fc3e4", "edgecolor": "#666666"})  # Da el mismo estilo a la diagonal.
for ax in axes.flat:  # Recorre los paneles de la matriz.
    ax.tick_params(labelsize=9)  # Reduce las etiquetas para que quepan en cada panel.
    ax.xaxis.label.set_fontsize(11)  # Ajusta los nombres de los ejes horizontales.
    ax.yaxis.label.set_fontsize(11)  # Ajusta los nombres de los ejes verticales.
plt.show()  # Muestra la matriz completa.

### Lectura de la matriz de dispersión

- En `Strength` frente a `Cement` se aprecia una **tendencia creciente**, pero con dispersión vertical: una misma cantidad de cemento no determina una única resistencia.
- `Age` presenta valores repetidos, visibles como bandas, y muchos casos se concentran en edades pequeñas respecto al máximo.
- Los ceros de algunos componentes generan acumulaciones de puntos. También hay relaciones y agrupaciones entre predictoras: no son ocho entradas completamente independientes entre sí.

**Implicación:** tiene sentido empezar con un modelo sencillo, pero la dispersión y los posibles cambios de pendiente aconsejan explorar otras variables y formas funcionales. La matriz no decide qué términos deben conservarse.

### 3.2.3. Relación entre resistencia y cemento: LOWESS

Los puntos representan las observaciones y la curva **LOWESS** resume la tendencia local sin imponer una recta. `fraccion` indica aproximadamente qué proporción de casos participa en cada ajuste local: una fracción pequeña sigue más detalles y ruido; una grande suaviza más.

La función siguiente reúne el gráfico y el suavizado. Si encuentra problemas numéricos, amplía el vecindario y muestra la fracción finalmente aplicada. No hace falta memorizar ese control: céntrate en la llamada y en la forma de la curva.

**Cómo interpretarlo:** un cambio de pendiente sugiere revisar la forma funcional. LOWESS no es el modelo final ni demuestra el efecto aislado del cemento: aquí aún no controlamos las otras variables.

In [ ]:
def dibujar_relacion(ax, x, y, nombre, fraccion=2/3):  # Dibuja los datos y su tendencia local en un eje.
    ax.scatter(x, y, s=30, color="#9fc3e4", edgecolor="#666666",  # Representa cada observación.
               linewidth=0.4, alpha=0.6)  # Reduce la superposición visual.
    aplicada = fraccion  # Empieza con el vecindario solicitado.
    while True:  # Reintenta el suavizado solo si hay problemas numéricos.
        with warnings.catch_warnings(record=True) as avisos:  # Recoge los avisos sin ocultarlos conceptualmente.
            warnings.simplefilter("always", RuntimeWarning)  # Detecta avisos de cálculo numérico.
            curva = lowess(y, x, frac=aplicada, it=3, delta=0.01 * np.ptp(x))  # Calcula el suavizado local robusto.
        numericos = any(issubclass(aviso.category, RuntimeWarning) for aviso in avisos)  # Comprueba si hubo avisos.
        if np.isfinite(curva).all() and not numericos:  # Acepta únicamente una curva con valores finitos.
            break  # Termina cuando el cálculo es válido.
        if aplicada >= 1:  # No puede ampliar más el vecindario.
            raise ValueError("No se puede estimar LOWESS con estos valores.")  # Informa del problema.
        aplicada = min(1, aplicada + 0.05)  # Amplía la fracción sin superar el total de observaciones.
    ax.plot(curva[:, 0], curva[:, 1], color="#800000", linewidth=1.8)  # Superpone la tendencia en granate.
    ax.set(xlabel=nombre, ylabel="Strength")  # Identifica los ejes.
    if aplicada > fraccion + 1e-8:  # Indica si fue necesario cambiar la fracción.
        ax.text(0.02, 0.98, f"Fracción aplicada: {aplicada:.2f}",  # Muestra el valor finalmente utilizado.
                transform=ax.transAxes, va="top", fontsize=11)  # Sitúa la anotación dentro del panel.

def explorar_relacion(variable, fraccion):  # Prepara una figura para el predictor elegido.
    fig, ax = plt.subplots(figsize=(16, 8))  # Crea la figura y el eje.
    dibujar_relacion(ax, datos[variable], datos["Strength"], variable, fraccion)  # Utiliza el entrenamiento.
    ax.set_title("Resistencia y " + variable)  # Identifica la relación explorada.
    fig.tight_layout()  # Ajusta márgenes.
    plt.show()  # Muestra el resultado.

In [ ]:
explorar_relacion("Cement", 2/3)  # Explora el cemento usando aproximadamente dos tercios de los casos por ajuste local.

### Lectura del suavizado

La curva aumenta con `Cement`, mientras los puntos se dispersan ampliamente alrededor de ella. La resistencia depende de más aspectos que la cantidad de cemento; la curva no predice con exactitud cada observación.

**Implicación:** una recta es un punto de partida razonable para esta relación marginal, pero después habrá que examinar los residuos y añadir otras predictoras.

#### Micropráctica: cambiar de predictor

Usa la llamada anterior para explorar `Age` con fracción `2/3`. Completa el nombre de la variable y compara ambas curvas: ¿la relación con la edad parece igual de recta? Describe la forma, no un efecto causal.

In [ ]:
try:
    variable_explorada = ...  # Completa con el nombre del predictor de edad.
    explorar_relacion(variable_explorada, 2/3)  # Reutiliza el suavizado del ejemplo anterior.
    # Compara la curva con la de Cement y redacta una conclusión sobre su forma.
except Exception as error:
    plt.close("all")  # Cierra una posible figura incompleta.
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### 3.2.4. Comparar las ocho predictoras

Repetimos los puntos y LOWESS para cada predictor frente a `Strength`. Observa cambios de pendiente, acumulaciones en cero y variaciones de la dispersión vertical.

**Qué estamos comparando:** relaciones **marginales**, una variable cada vez. Una curva puede cambiar cuando ajustemos todas las predictoras juntas; no representa un efecto aislado.

Después de ejecutar, elige un panel que parezca aproximadamente lineal y otro que muestre curvatura. Esas observaciones orientarán la revisión de residuos, no una selección automática de variables.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))  # Reserva un panel por cada predictora.
for ax, variable in zip(axes.flat, predictores):  # Empareja cada eje con una variable de entrada.
    dibujar_relacion(ax, datos[variable], datos["Strength"], variable)  # Dibuja datos y LOWESS con la misma función.
    _ = ax.set_title(variable, fontsize=16)  # Identifica el predictor.
    ax.xaxis.label.set_fontsize(13)  # Ajusta las etiquetas horizontales.
    ax.yaxis.label.set_fontsize(13)  # Ajusta las etiquetas verticales.
    ax.tick_params(labelsize=11)  # Mantiene legibles las escalas de los paneles.
fig.tight_layout()  # Evita superposiciones.
plt.show()  # Muestra las ocho relaciones marginales.

### Lectura conjunta de los ocho paneles

`Cement` y `Superplasticizer` muestran tendencias crecientes. En `Age`, la curva aumenta con rapidez en las edades pequeñas y después se aplana; `Water` también presenta un cambio de pendiente. Las acumulaciones en cero de algunos componentes quedan claramente visibles.

**Implicación para el modelo:** una única recta para cada entrada puede dejar curvatura sin explicar. Empezaremos por una regresión sencilla para entender coeficientes y residuos, y después comprobaremos si un modelo multivariable y términos cuadráticos mejoran el análisis.

Estas formas son marginales y dependen del suavizado y de los datos disponibles; no deben convertirse directamente en afirmaciones causales.

## 3.3. Modelo lineal simple

### 3.3.1. Ajustar la resistencia a partir del cemento

Empezamos con una sola predictora:

$$
Strength_i=\beta_0+\beta_1\,Cement_i+\varepsilon_i,
\qquad \widehat y_i=\widehat\beta_0+\widehat\beta_1\,Cement_i.
$$

**Mínimos cuadrados** elige los coeficientes que minimizan $\sum_i(y_i-\widehat y_i)^2$. `sm.add_constant` añade una columna de unos para estimar el intercepto; `sm.OLS(...).fit()` ajusta el modelo. `missing="raise"` evita ignorar filas con ausentes sin avisar.

La pendiente describe un cambio promedio estimado por unidad de cemento **sin controlar todavía los demás componentes**. La inferencia clásica sobre los coeficientes supone una forma media adecuada, errores independientes y varianza constante; la normalidad de los errores fundamenta los contrastes e intervalos exactos en muestras pequeñas.

In [ ]:
y = datos["Strength"]  # Define la respuesta que queremos predecir.
X_simple = sm.add_constant(datos[["Cement"]], has_constant="add")  # Construye el diseño con intercepto y cemento.
mod_lm0 = sm.OLS(y, X_simple, missing="raise").fit()  # Estima los coeficientes por mínimos cuadrados.
display(mod_lm0.summary())  # Muestra coeficientes, incertidumbre, contrastes y medidas de ajuste.

### Lectura del modelo simple

La recta ajustada es aproximadamente

$$
\widehat{Strength}=12{,}62+0{,}0812\,Cement.
$$

- **Pendiente:** una unidad adicional de `Cement` se asocia con unos **0,0812 MPa** más de resistencia media estimada en este modelo. No es un efecto causal ni ajustado por las demás variables.
- **Intercepto:** corresponde a `Cement = 0`, fuera del rango observado (102–540); no debe interpretarse como una receta real validada.
- **Inferencia:** el contraste $H_0:\beta_1=0$ frente a $H_1:\beta_1\ne0$ da un valor p muy pequeño. El `0.000` del resumen es redondeo, no una probabilidad exactamente cero. `std err` es el error estándar del coeficiente.
- **Ajuste:** $R^2\approx0{,}252$ indica que la recta explica aproximadamente el **25,2 % de la variación muestral de `Strength`** respecto a un modelo con solo intercepto. No significa un 25,2 % de predicciones correctas.

**Implicación:** hay asociación, pero queda bastante variación sin explicar. La significación estadística no garantiza buenas predicciones.

### 3.3.2. Residuos del modelo

Definimos $e_i=y_i-\widehat y_i$. Un residuo positivo indica que el modelo subestimó la resistencia; uno negativo, que la sobreestimó. `abs()` compara magnitudes sin importar el signo y `nlargest(3)` localiza los tres casos más alejados de la recta.

In [ ]:
mod_lm0.resid.abs().nlargest(3)  # Muestra las etiquetas de los tres mayores errores absolutos de ajuste, en MPa.

Los casos **99, 263 y 113** tienen los mayores residuos absolutos en este ajuste, de aproximadamente **41,29, 41,29 y 38,31 MPa**. Son errores de ajuste sobre entrenamiento, no errores medidos en test ni pruebas de que los datos sean incorrectos.

### 3.3.3. Datos y recta ajustada

Superponemos la recta sobre los puntos. Para dibujarla, creamos 100 cantidades de cemento entre el mínimo y el máximo del entrenamiento y usamos la misma estructura de diseño, con constante y `Cement`.

**Cómo leerlo:** la separación vertical entre un punto y la recta es su residuo. La recta resume una media estimada; no tiene que pasar por cada observación y aquí no se representa todavía un intervalo de predicción.

In [ ]:
rejilla = pd.DataFrame({"Cement": np.linspace(datos["Cement"].min(), datos["Cement"].max(), 100)})  # Usa el rango observado.
fig, ax = plt.subplots(figsize=(16, 8))  # Prepara la figura.
_ = ax.scatter(datos["Cement"], y, s=40, color="#9fc3e4", alpha=0.65,  # Representa las observaciones de entrenamiento.
               edgecolor="#666666", linewidth=0.6)  # Facilita distinguir puntos.
_ = ax.plot(rejilla["Cement"], mod_lm0.predict(sm.add_constant(rejilla, has_constant="add")),  # Predice con constante y Cement.
            color="#800000", linewidth=1.8)  # Dibuja la recta ajustada.
_ = ax.set(xlabel="Cement", ylabel="Strength (MPa)", title="Modelo lineal simple")  # Identifica ejes y unidades de la respuesta.
fig.tight_layout()  # Ajusta márgenes.
plt.show()  # Muestra datos y predicción media.

#### Micropráctica: interpretar un cambio de cemento

Los coeficientes están en `mod_lm0.params`; la pendiente se recupera con `mod_lm0.params['Cement']`. En la celda se muestra cómo calcular el cambio de resistencia media para **10 unidades** adicionales de cemento. Completa el mismo cálculo para **50 unidades**.

Interpreta el resultado en MPa y explica por qué no es una predicción garantizada para una pieza individual ni un efecto causal. Mantén la comparación dentro del rango observado.

In [ ]:
try:
    cambio_10 = 10 * mod_lm0.params['Cement']  # Ejemplo: multiplica el cambio de entrada por la pendiente.
    cambio_50 = ...  # Repite el patrón para 50 unidades adicionales de cemento.
    print(f'Cambio para 10 unidades: {cambio_10:.2f} MPa; para 50: {cambio_50:.2f} MPa')
    # Redacta una interpretación de cambio medio, no de certeza individual.
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

## 3.4. Modelo lineal multivariable

### 3.4.1. Incluir varias predictoras

Ahora modelizamos la respuesta con todas las entradas:

$$
y_i=\beta_0+\sum_{j=1}^{p}\beta_jx_{ij}+\varepsilon_i.
$$

Cada coeficiente describe el cambio estimado al variar una predictora **manteniendo constantes las demás columnas del modelo**. No equivale a las asociaciones marginales de los gráficos; si las entradas están relacionadas, los coeficientes pueden ser inestables.

Añadimos además **dos columnas de ruido artificial**, `Ruido1` y `Ruido2`, solo a `X_lineal`. La semilla 42 permite repetir exactamente el ejemplo. No alteramos `datos` ni `predictores`.

In [ ]:
azar = np.random.default_rng(42)  # Fija la semilla del ruido artificial.
X_lineal = sm.add_constant(datos[predictores], has_constant="add")  # Añade el intercepto a las ocho entradas.
X_lineal["Ruido1"] = azar.standard_normal(len(datos))  # Crea la primera columna de ruido normal.
X_lineal["Ruido2"] = azar.standard_normal(len(datos))  # Crea la segunda columna con el mismo generador.
mod_lm1 = sm.OLS(y, X_lineal, missing="raise").fit()  # Ajusta el modelo completo sobre el entrenamiento.
# Para probar cambios posteriores, usa X_lineal.copy() y conserva esta matriz original.

In [ ]:
display(mod_lm1.summary())  # Muestra los coeficientes condicionados por las otras entradas y el ajuste global.

### Lectura del modelo multivariable

El $R^2$ de entrenamiento aumenta a aproximadamente **0,626**, frente a 0,252 en el modelo simple. Esto describe una mejora de ajuste en las mismas 706 filas, **no una evaluación con datos nuevos**.

La pendiente de `Cement` pasa a aproximadamente **0,1252**: ahora está condicionada por las otras columnas. Que cambie respecto a la regresión simple ilustra la diferencia entre una relación marginal y una ajustada.

Los p-valores de `Ruido1` y `Ruido2` son aproximadamente **0,393 y 0,447**. No hay evidencia suficiente contra un coeficiente nulo al 5 % dentro de este modelo, pero un p-valor grande no demuestra que una variable sea inútil en toda especificación.

### 3.4.2. Coeficientes y p-valores

La siguiente operación elimina `const` y ordena los coeficientes. Es un **ejemplo de selección y ordenación de una serie** que reutilizaremos en la micropráctica, no una medida automática de importancia.

In [ ]:
mod_lm1.params.drop(index="const").sort_values(ascending=True)  # Quita el intercepto y ordena los coeficientes de menor a mayor.

#### Micropráctica: primera candidata a revisar

Reutiliza el patrón anterior: cambia `params` por `pvalues`, excluye `const` y ordena **de mayor a menor**. Completa los dos huecos y anota qué variable revisarías primero.

Después compara tu candidata con el primer paso de la selección por AIC. **Un p-valor alto no basta para eliminar una variable:** hay que examinar el cambio de ajuste, los demás coeficientes y el sentido del modelo.

In [ ]:
try:
    # Ordena los p-valores de las predictoras, excluyendo el término constante.
    p_sin_constante = mod_lm1.pvalues.drop(index=...)  # Completa el nombre del término constante.
    p_ordenados = p_sin_constante.sort_values(ascending=...)  # Elige el orden para que el p-valor mayor aparezca primero.
    print(p_ordenados)  # Identifica qué predictor revisarías antes y compáralo con el primer paso AIC.
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

## 3.5. Selección de variables por AIC

### 3.5.1. Eliminación hacia atrás

El AIC combina ajuste y penalización por complejidad:

$$
AIC=-2\,\widehat\ell+2k,
$$

con $\widehat\ell$ la log-verosimilitud maximizada y $k$ el número de parámetros penalizados. **Menor AIC es preferible** al comparar modelos sobre la misma respuesta y las mismas filas; el valor aislado no es una nota de calidad.

`seleccion_atras` sigue estos pasos:

1. Parte del modelo completo.
2. Prueba retirar cada grupo de términos, uno cada vez.
3. Acepta la retirada con menor AIC **solo si mejora el AIC actual**.
4. Se detiene cuando ninguna retirada lo reduce.

La función devuelve el modelo, los términos conservados y la tabla de pasos. No fuerza la eliminación del ruido ni garantiza el mejor modelo entre todas las combinaciones posibles. La selección usa **solo entrenamiento** y sus p-valores posteriores no incorporan la incertidumbre del proceso de selección.

In [ ]:
from s3_seleccion_atras import seleccion_atras  # Importa el procedimiento de selección por AIC encapsulado en su script.

In [ ]:
grupos_lineales = {variable: [variable] for variable in X_lineal.columns if variable != "const"}  # Un grupo por predictora.
mod_lm2, seleccion_lineal, pasos_lineales = seleccion_atras(y, X_lineal, grupos_lineales)  # Selecciona sin retirar el intercepto.
display(pasos_lineales)  # Muestra la retirada, el AIC y los términos restantes en cada paso.

### Lectura de la selección lineal

El procedimiento retira primero **`Ruido2`** y después **`Ruido1`**. El AIC disminuye de aproximadamente **5282,29 a 5279,64**, y se conservan las ocho predictoras reales.

**Qué significa:** en cada paso la simplificación aceptada compensa la pérdida de ajuste según el criterio AIC. No significa que cualquier predictor artificial se elimine siempre ni que todas las variables conservadas tengan p-valores inferiores a 0,05.

**Siguiente comprobación:** un AIC menor no demuestra linealidad, normalidad, varianza constante o independencia. Antes de confiar en este modelo hay que revisar sus residuos.

#### Micropráctica: cuantificar la simplificación

Calcula `mod_lm1.aic - mod_lm2.aic` y lista los términos de `grupos_lineales` que no aparecen en `seleccion_lineal`. Completa los huecos reutilizando una resta.

Comprueba la reducción con la tabla de pasos y redacta qué se ha simplificado. Un resultado positivo indica **reducción** de AIC, no mejora demostrada en test.

In [ ]:
try:
    # Compara el AIC antes y después de la selección y lista los términos retirados.
    mejora_aic = ...  # Resta los atributos .aic de mod_lm1 y mod_lm2.
    print(f'Reducción de AIC: {mejora_aic:.2f}')  # Un valor positivo indica una reducción del AIC.
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

In [ ]:
display(mod_lm2.summary())  # Revisa el modelo lineal sin las dos columnas de ruido.

## 3.6. Diagnóstico del modelo lineal

### 3.6.1. Cuatro vistas de los residuos

El modelo seleccionado conserva $R^2\approx0{,}625$, pero un buen ajuste no confirma sus supuestos. `diagnosticos` reúne cuatro paneles:

| Panel | Qué buscamos | Qué puede sugerir una pauta |
|---|---|---|
| Residuos frente a ajustados | Banda alrededor de cero, sin curva | Forma funcional omitida |
| Q–Q de residuos estandarizados | Puntos aproximadamente alineados | Colas distintas de la normal |
| Escala y localización | Dispersión aproximadamente uniforme | Varianza no constante |
| Residuos y apalancamiento | Casos alejados y curvas de Cook | Observaciones potencialmente influyentes |

La función está en un módulo auxiliar para centrarnos en **leer los gráficos**, no en programarlos. Las líneas son referencias, no reglas de eliminación.

**Importante:** diagnosticar los residuos de entrenamiento no equivale a evaluar la predicción en test. Ningún panel demuestra por sí solo independencia ni validez completa del modelo.

In [ ]:
from s3_diagnosticos import diagnosticos  # Importa la función que genera los cuatro gráficos de diagnóstico.

In [ ]:
diagnosticos(mod_lm2, "Diagnóstico del modelo lineal seleccionado")  # Examina forma, colas, dispersión e influencia.

### Lectura de los diagnósticos lineales

- **Forma:** la curva de residuos frente a ajustados sube en la zona central y vuelve a bajar. No es una banda completamente sin estructura alrededor de cero.
- **Normalidad:** el Q–Q se aproxima bastante a la recta en el centro, con desviaciones en los extremos. No demuestra por sí solo normalidad.
- **Dispersión:** la curva de escala y localización aumenta con el ajuste, lo que sugiere revisar la varianza constante.
- **Influencia:** hay casos con apalancamiento mayor que el grueso de la muestra; se examinarán después con sus datos originales.

**Implicación:** conviene revisar la forma funcional y la dispersión antes de interpretar los intervalos clásicos como plenamente fiables. El siguiente gráfico aborda el orden de los residuos, con una limitación importante sobre qué representa ese orden.

### 3.6.2. Residuos en el orden del archivo

Representamos los residuos en el **orden de las filas de entrenamiento**. Rachas del mismo signo, tendencias o ciclos pueden revelar estructura que el modelo no recoge.

**Límite:** `observacion` es una etiqueta asignada por nosotros, no una fecha. Si las filas se mezclaron, este gráfico no diagnostica dependencia temporal; ni un gráfico sin pauta ni un estadístico calculado sobre un orden artificial demuestra independencia.

Para valorar ese supuesto hacen falta datos sobre el muestreo y, cuando proceda, el orden real de observación.

In [ ]:
def grafico_orden_residuos(modelo, titulo):  # Muestra los residuos según su posición en el archivo.
    fig, ax = plt.subplots(figsize=(16, 6))  # Prepara la figura.
    orden = np.arange(1, len(modelo.resid) + 1)  # Numera posiciones, sin suponer fechas.
    _ = ax.plot(orden, modelo.resid, color="#34699a", alpha=0.7, linewidth=1.1,  # Une residuos consecutivos.
                marker=".", markersize=4)  # Marca cada observación.
    _ = ax.axhline(0, color="#800000", linestyle="--", linewidth=1.8)  # Facilita comparar signos.
    _ = ax.set(xlabel="Orden de las filas en entrenamiento", ylabel="Residuo", title=titulo)  # Identifica los ejes.
    fig.tight_layout()  # Ajusta márgenes.
    plt.show()  # Muestra el gráfico.

grafico_orden_residuos(mod_lm2, "Residuos del modelo lineal en el orden de las filas")  # Aplica la revisión al modelo seleccionado.

### Lectura del gráfico en orden

Se observan zonas con diferencias de signo y amplitud, especialmente entre las primeras filas. Esto justifica revisar cómo se organizaron los datos, pero no permite llamar a esas zonas «periodos temporales» sin información adicional.

#### Micropráctica: localizar residuos extremos

Reutiliza `mod_lm0.resid.abs().nlargest(3)`, cambiando el modelo por `mod_lm2`. Completa el número de casos y compara las etiquetas obtenidas con las del modelo simple.

¿Por qué un residuo grande puede cambiar al incluir más predictoras? Explica por qué conviene examinar las observaciones antes de decidir si son errores.

In [ ]:
try:
    # Encuentra las tres observaciones con residuos de mayor magnitud en el modelo lineal.
    residuos_abs = mod_lm2.resid.abs()  # Convierte cada residuo en su magnitud.
    top_residuos = residuos_abs.nlargest(...)  # Completa el número de observaciones que vas a listar.
    print(top_residuos)  # Muestra las etiquetas de fila y el tamaño de esos residuos.
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

## 3.7. Buscar la forma que falta

### 3.7.1. Residuos frente a predictoras

Repetimos los gráficos exploratorios, pero ahora el eje vertical contiene **residuos de `mod_lm2`**. Si el modelo recoge bien la forma de la relación, no debería quedar una curva persistente alrededor de cero.

Una tendencia residual puede sugerir curvatura omitida, grupos distintos o una transformación que merezca estudio. No identifica por sí sola la solución ni un efecto causal.

La función reutiliza `dibujar_relacion`. Compara especialmente `Age` con su gráfico marginal anterior: ¿qué patrón sigue sin explicar el término lineal?

In [ ]:
def residuos_predictoras(modelo, tabla, variables):  # Examina la parte de la respuesta que el modelo no explica.
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))  # Prepara hasta ocho paneles.
    for ax, variable in zip(axes.flat, variables):  # Recorre las entradas originales.
        dibujar_relacion(ax, tabla[variable], modelo.resid, variable)  # Suaviza residuos, no Strength.
        ax.set(title=variable, ylabel="Residuos")  # Identifica el contenido del eje vertical.
        ax.axhline(0, color="#666666", linestyle="--", linewidth=1.2)  # Marca la ausencia de error medio local.
    for ax in list(axes.flat)[len(variables):]:  # Localiza paneles sobrantes.
        ax.set_visible(False)  # Oculta los paneles vacíos.
    for ax in axes.flat:  # Ajusta la legibilidad de los paneles utilizados.
        if ax.get_visible():
            ax.title.set_fontsize(16)  # Ajusta títulos.
            ax.xaxis.label.set_fontsize(13)  # Ajusta etiquetas horizontales.
            ax.yaxis.label.set_fontsize(13)  # Ajusta etiquetas verticales.
            ax.tick_params(labelsize=11)  # Ajusta marcas de escala.
    fig.tight_layout()  # Evita solapamientos.
    plt.show()  # Muestra el resultado.

residuos_predictoras(mod_lm2, datos, predictores)  # Busca patrones que no recoge el modelo lineal.

### Lectura de los residuos frente a predictoras

En `Age` queda una curva marcada: los residuos aumentan en edades intermedias y disminuyen en las más altas. También se ven cambios de tendencia en otras entradas. Esto sugiere que los términos lineales no recogen toda la forma.

### 3.7.2. Apoyar la exploración con pruebas de curvatura

Para cada término solo lineal, añadimos provisionalmente su cuadrado al **mismo modelo** y comparamos dos ajustes anidados:

$$
H_0:\beta_{\text{cuadrado}}=0,\qquad H_1:\beta_{\text{cuadrado}}\ne0.
$$

La prueba F evalúa si el término adicional reduce suficientemente el error de ajuste bajo los supuestos clásicos. También se prueba el cuadrado del valor ajustado, una comprobación global de no aditividad etiquetada como Tukey.

**Qué dibuja cada panel:** cuando se prueba un cuadrado, se muestran los residuos del **modelo ampliado provisionalmente**, para ver qué pauta queda tras añadirlo. Los términos que ya incluyen grado 2 se dibujan con los residuos del **modelo original**, sin repetir el mismo cuadrado. Ninguna ampliación sustituye al modelo principal.

**Un valor p pequeño es una señal exploratoria, no una orden de transformar:** hay varias pruebas, supuestos y decisiones que revisar junto con los gráficos.

In [ ]:
def residuos_por_termino(modelo, tabla, grupos):  # Dibuja residuos y prueba ampliaciones cuadráticas de un grado de libertad.
    variables = list(grupos)  # Obtiene los términos activos del modelo.
    paneles = [(variable, tabla[variable], len(grupos[variable]) == 1) for variable in variables]  # Prueba solo términos lineales.
    paneles.append(("Ajustados (Tukey)", modelo.fittedvalues, True))  # Añade la comprobación global.
    filas = int(np.ceil(len(paneles) / 3))  # Calcula las filas necesarias.
    fig, axes = plt.subplots(filas, 3, figsize=(18, 4.5 * filas), squeeze=False)  # Crea los paneles.
    resultados = []  # Guardará F y valor p de cada comparación.
    for ax, (nombre, valores, probar) in zip(axes.flat, paneles):  # Recorre términos y valores ajustados.
        ax.set(title=nombre, ylabel="Residuos")  # Identifica el panel.
        ax.axhline(0, color="#666666", linestyle="--", linewidth=1.2)  # Marca la referencia cero.
        ax.title.set_fontsize(18)  # Ajusta el título.
        ax.xaxis.label.set_fontsize(14)  # Ajusta la etiqueta horizontal.
        ax.yaxis.label.set_fontsize(14)  # Ajusta la etiqueta vertical.
        ax.tick_params(labelsize=12)  # Ajusta las escalas.
        if probar:  # No duplica cuadrados ya incluidos en el diseño.
            x = np.asarray(valores, dtype=float)  # Convierte los valores en un vector numérico.
            x = (x - x.mean()) / x.std()  # Centra y escala para estabilizar el cálculo.
            ampliada = np.column_stack([modelo.model.exog, x ** 2])  # Añade solo el término cuadrático.
            ampliado = sm.OLS(modelo.model.endog, ampliada).fit()  # Ajusta sobre la misma respuesta y filas.
            f, p, _ = ampliado.compare_f_test(modelo)  # Compara el modelo ampliado con el original.
            dibujar_relacion(ax, valores, ampliado.resid, nombre)  # Muestra los residuos tras la ampliación provisional.
            resultados.append({"Término": nombre, "F": f, "p-valor": p})  # Guarda la prueba.
        else:  # Los términos ya cuadráticos se dibujan sin una prueba redundante.
            dibujar_relacion(ax, valores, modelo.resid, nombre)  # Muestra sus residuos en el modelo original.
        ax.set_ylabel("Residuos")  # Restablece la etiqueta tras el suavizado auxiliar.
    for ax in list(axes.flat)[len(paneles):]:  # Localiza paneles no utilizados.
        ax.set_visible(False)  # Oculta los paneles vacíos.
    fig.tight_layout()  # Ajusta márgenes.
    plt.show()  # Muestra los gráficos.
    return pd.DataFrame(resultados).set_index("Término")  # Devuelve la tabla de comparaciones.

In [ ]:
pruebas_lineales = residuos_por_termino(mod_lm2, datos, {v: grupos_lineales[v] for v in seleccion_lineal})  # Examina los términos conservados.
display(pruebas_lineales)  # Muestra las comparaciones F de las ampliaciones cuadráticas.

### Lectura de las pruebas de curvatura

La ampliación cuadrática de `Age` da $F\approx324{,}97$ y un valor p extremadamente pequeño; concuerda con su curva residual marcada. También aparecen señales de curvatura en otros términos y en la comprobación global. Para `CoarseAggregate`, el valor p es aproximadamente **0,898**, sin evidencia de mejora cuadrática en esta comparación.

**Implicación:** exploraremos un diseño con curvatura para varias entradas y mantendremos `CoarseAggregate` lineal. No se está afirmando que el grado 2 sea la forma definitiva ni que cada p-valor aislado determine el modelo; la selección y los diagnósticos siguen siendo necesarios.

## 3.8. Modelo con términos polinómicos

### 3.8.1. Permitir curvatura de grado 2

Para siete predictoras construimos dos columnas:

$$
z_j=\frac{x_j-\bar x_{j,\mathrm{train}}}{s_{j,\mathrm{train}}},
\qquad z_j\quad\text{y}\quad z_j^2.
$$

La escala se calcula aquí con `std(ddof=0)`; sirve para estandarizar, no es un error estándar. `CoarseAggregate` permanece lineal en su escala original. El modelo es **lineal en sus coeficientes**, aunque represente curvas respecto a las entradas; no incluye interacciones entre variables.

Centrar y escalar mejora la estabilidad numérica sin cambiar el conjunto de curvas cuadráticas representables. Cambia la interpretación individual de los coeficientes, por lo que no deben compararse directamente con los coeficientes en escala original.

**Regla para predecir:** aprendemos las medias y escalas solo con entrenamiento y las guardamos en `parametros_polinomicos`. En test se reutilizan; no se recalculan.

Los grupos vinculan las columnas lineal y cuadrática de cada variable para conservarlas o retirarlas **juntas** durante la selección por AIC.

In [ ]:
def diseno_polinomico(tabla, parametros=None):  # Construye el mismo tipo de columnas al ajustar y al predecir.
    if parametros is None:  # Aprende parámetros solo si no se proporcionan los del entrenamiento.
        parametros = {v: (tabla[v].mean(), tabla[v].std(ddof=0))  # Guarda centro y escala de cada entrada cuadrática.
                      for v in predictores if v != "CoarseAggregate"}  # Deja el árido grueso en escala original.
    matriz = pd.DataFrame({"const": 1.0}, index=tabla.index)  # Conserva las etiquetas y añade el intercepto.
    for variable in predictores:  # Recorre las ocho entradas, nunca la respuesta.
        if variable == "CoarseAggregate":  # Esta entrada permanece lineal.
            matriz[variable] = tabla[variable]  # Copia sus valores originales.
        else:
            centro, escala = parametros[variable]  # Recupera los parámetros aprendidos.
            z = (tabla[variable] - centro) / escala  # Estandariza con el mismo centro y escala.
            matriz[variable + "_1"] = z  # Añade el término de primer grado.
            matriz[variable + "_2"] = z ** 2  # Añade el término de segundo grado.
    return matriz, parametros  # Devuelve el diseño y los parámetros necesarios para test.

In [ ]:
X_polinomico, parametros_polinomicos = diseno_polinomico(datos)  # Aprende centros y escalas con entrenamiento.
grupos_polinomicos = {  # Define qué columnas se conservan o retiran juntas.
    variable: [variable] if variable == "CoarseAggregate" else [variable + "_1", variable + "_2"]  # Uno o dos coeficientes por término.
    for variable in predictores  # Incluye solo las ocho predictoras reales, sin ruido.
}
mod_lm4 = sm.OLS(y, X_polinomico, missing="raise").fit()  # Ajusta el modelo polinómico completo.
display(mod_lm4.summary())  # Muestra su ajuste antes de seleccionar términos.

### Lectura del modelo polinómico completo

El diseño tiene **15 columnas predictoras más el intercepto**: dos por cada una de las siete variables cuadráticas y una para `CoarseAggregate`. El $R^2$ de entrenamiento aumenta a aproximadamente **0,780**.

Una columna como `Cement_2` es el cuadrado de la variable **estandarizada**, no del cemento original. Su coeficiente no debe leerse como un cambio constante por unidad de cemento: la pendiente de una curva cuadrática depende del punto de la curva.

### 3.8.2. Seleccionar términos completos

Aplicamos el mismo AIC, pero retirando grupos: si se elimina una variable cuadrática, desaparecen sus dos columnas. Un p-valor alto de una columna no obliga a retirar todo el término ni a conservar solo su otra columna.

In [ ]:
mod_lm5, seleccion_polinomica, pasos_polinomicos = seleccion_atras(y, X_polinomico, grupos_polinomicos)  # Retira grupos solo si reduce AIC.
display(pasos_polinomicos)  # Muestra qué términos se conservaron y la evolución del criterio.

In [ ]:
display(mod_lm5.summary())  # Revisa los coeficientes del modelo polinómico seleccionado con AIC.

### Lectura de la selección polinómica

Se retira **`CoarseAggregate`** y el AIC baja de aproximadamente **4918,35 a 4916,43**. Quedan siete variables, cada una con dos columnas, y el $R^2$ se mantiene alrededor de **0,780**. Los valores son comparables con los modelos en `Strength` porque usan las mismas 706 filas y la misma respuesta.

**Implicación:** el ajuste es mejor que el lineal según estos indicadores de entrenamiento, pero aún hay que revisar colinealidad, residuos y predicción fuera de la muestra. Las columnas con p-valores altos siguen dentro de sus grupos; el procedimiento no seleccionó coeficientes individuales.

#### Micropráctica: reconstruir un término cuadrático

Usa los parámetros guardados de `Cement` para calcular su versión estandarizada y elevarla al cuadrado. Completa los dos huecos y compara los primeros valores con `X_polinomico['Cement_2']`.

¿Por qué no sería correcto usar otra media y otra escala al construir esa columna para test?

In [ ]:
try:
    # Reconstruye la columna cuadrática estandarizada de Cement.
    centro_cement, escala_cement = parametros_polinomicos[...]  # Completa el nombre del predictor en el diccionario.
    z_cement = (datos['Cement'] - centro_cement) / escala_cement  # Estandariza usando solo los parámetros aprendidos en entrenamiento.
    cement_cuadrado = ...  # Completa la potencia para obtener el término de segundo grado.
    print(cement_cuadrado.head())  # Inspecciona algunos valores calculados.
    print(X_polinomico['Cement_2'].head())  # Compáralos con la columna creada por diseno_polinomico.
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

## 3.9. Colinealidad y diagnóstico polinómico

### 3.9.1. Información redundante entre términos

La **colinealidad** aparece cuando varias columnas aportan información parecida. Puede aumentar la incertidumbre e inestabilidad de los coeficientes aunque el modelo prediga razonablemente; no es lo mismo que una curva en los residuos.

`gvif` calcula un factor por **término**, usando la correlación entre coeficientes estimados y excluyendo la constante. Para un término de una columna coincide con el VIF habitual; en términos de varias columnas también muestra

$$
GVIF^{1/(2\,Df)},
$$

con `Df` el número de coeficientes del término. Esta medida ajustada facilita comparar tamaños de grupos; no debe aplicársele sin más un umbral pensado para el VIF bruto de una columna.

**Cómo usarlo:** valores mayores indican más redundancia relativa. Son señales para revisar la interpretación y estabilidad, no reglas automáticas para borrar variables.

In [ ]:
def gvif(modelo, grupos):  # Calcula colinealidad por términos de una o varias columnas.
    covarianza = modelo.cov_params().drop(index="const", columns="const")  # Excluye el intercepto.
    desviaciones = np.sqrt(np.diag(covarianza))  # Recupera los errores estándar de los coeficientes.
    correlacion = covarianza / np.outer(desviaciones, desviaciones)  # Convierte covarianzas en correlaciones.
    columnas = list(correlacion.columns)  # Guarda el orden de las columnas del diseño.

    def logdet(matriz):  # Usa log-determinantes para estabilizar los cocientes.
        if matriz.size == 0:  # El complemento puede estar vacío.
            return 0.0  # El determinante de la matriz vacía se toma como uno.
        signo, valor = np.linalg.slogdet(matriz)  # Calcula signo y logaritmo del determinante.
        if signo <= 0:  # Detecta una matriz no válida para esta medida.
            raise ValueError("La matriz de coeficientes es singular.")
        return valor  # Devuelve el log-determinante.

    total = logdet(correlacion.to_numpy())  # Referencia de la matriz completa.
    resultados = []  # Reunirá una fila por término.
    for variable, terminos in grupos.items():  # Trata conjuntamente las columnas de cada entrada.
        dentro = [columnas.index(termino) for termino in terminos]  # Posiciones del grupo.
        fuera = [i for i in range(len(columnas)) if i not in dentro]  # Posiciones del resto de términos.
        matriz = correlacion.to_numpy()  # Trabaja con la matriz numérica de correlación.
        valor = np.exp(logdet(matriz[np.ix_(dentro, dentro)]) + logdet(matriz[np.ix_(fuera, fuera)]) - total)  # Calcula GVIF.
        gl = len(dentro)  # Cuenta los coeficientes del término.
        resultados.append({"Término": variable, "GVIF": valor, "Df": gl,  # Guarda factor bruto y tamaño.
                           "GVIF^(1/(2·Df))": valor ** (1 / (2 * gl))})  # Añade la medida ajustada.
    return pd.DataFrame(resultados).set_index("Término")  # Devuelve una tabla para comparar grupos.

In [ ]:
grupos_seleccionados = {variable: grupos_polinomicos[variable] for variable in seleccion_polinomica}  # Usa solo términos activos.
tabla_gvif_modelo = gvif(mod_lm5, grupos_seleccionados)  # Calcula los factores de colinealidad por grupo.
display(tabla_gvif_modelo)  # Compara GVIF bruto, grados de libertad y medida ajustada.

### Lectura de la tabla GVIF

Todos los términos seleccionados tienen **dos grados de libertad**. Las mayores medidas ajustadas corresponden a `Superplasticizer` y `FlyAsh`, alrededor de **1,49**; `Age` presenta aproximadamente **1,05**.

**Implicación:** hay distinta redundancia entre grupos, pero esta tabla no obliga a retirar los dos primeros términos ni garantiza que sus coeficientes sean estables en cualquier muestra. Su importancia predictiva es una cuestión diferente.

#### Micropráctica: ordenar la colinealidad

Reutiliza `gvif(...)` y `sort_values(...)` para ordenar `GVIF^(1/(2·Df))` de **mayor a menor**. Completa la columna y el sentido del orden. Explica qué revisarías primero y por qué no lo eliminarías automáticamente.

In [ ]:
try:
    # Ordena la tabla GVIF por su medida ajustada para identificar los términos más colineales.
    tabla_gvif = gvif(mod_lm5, grupos_seleccionados)  # Calcula los factores para los términos seleccionados.
    orden_gvif = tabla_gvif.sort_values(by=..., ascending=...)  # Completa la columna y el orden descendente.
    display(orden_gvif)  # Presenta los términos desde mayor hasta menor colinealidad ajustada.
    # Usa la tabla para decidir qué término convendría inspeccionar con más detalle.
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### 3.9.2. Repetir los diagnósticos

Aplicamos los mismos gráficos a `mod_lm5` y lo comparamos con **`mod_lm2`**. Revisa si se reduce la curva en residuos frente a ajustados, si la dispersión es más uniforme y si siguen destacando casos en las colas o por apalancamiento.

Después volvemos a examinar los residuos por término. Los grupos cuadráticos no se prueban añadiendo otra vez el mismo grado 2; sí se mantiene la comprobación global de no aditividad.

**Criterio de lectura:** mejorar un patrón no confirma todos los supuestos. La comparación posterior de errores absolutos sigue siendo sobre entrenamiento; el test se reserva para el final.

In [ ]:
diagnosticos(mod_lm5, "Diagnóstico del modelo polinómico seleccionado")  # Repite los cuatro paneles para comparar patrones.
grafico_orden_residuos(mod_lm5, "Residuos del modelo polinómico en el orden de las filas")  # Conserva la cautela sobre el orden.

In [ ]:
pruebas_polinomicas = residuos_por_termino(mod_lm5, datos, grupos_seleccionados)  # Dibuja cada grupo y prueba la comprobación global.
display(pruebas_polinomicas)  # Los términos ya cuadráticos no generan una prueba redundante de grado 2.

### Lectura del diagnóstico polinómico

La curva de residuos frente a ajustados queda mucho más cerca de cero que en el modelo lineal. La comprobación global da $F\approx0{,}927$ y $p\approx0{,}336$: no se rechaza esa ampliación nula al 5 %, pero esto **no demuestra que la forma sea perfecta**.

Persisten desviaciones en las colas del Q–Q, una tendencia creciente en escala y localización y algunos casos con apalancamiento alto. En los residuos por `Age` aún queda estructura local. Mejorar el ajuste no ha resuelto todos los aspectos del diagnóstico.

#### Micropráctica: comparar errores absolutos de ajuste

Calcula la mediana de los residuos **en valor absoluto** de `mod_lm2` y `mod_lm5`. Reutiliza `abs()` y el percentil `50%` de `describe()`; completa la segunda expresión siguiendo el ejemplo de la primera.

¿Cuál describe menor error absoluto central en estas observaciones? Ambos resultados están en MPa y corresponden a **entrenamiento**, no a capacidad predictiva demostrada en test.

In [ ]:
try:
    # Usa el valor 50 % del resumen descriptivo para obtener la mediana de cada magnitud residual.
    mediana_lm2 = mod_lm2.resid.abs().describe()["50%"]  # Reutiliza abs() del ejercicio de residuos extremos.
    mediana_lm5 = ...  # Repite el patrón anterior sustituyendo mod_lm2 por mod_lm5.
    print(f'mod_lm2: {mediana_lm2:.3f}; mod_lm5: {mediana_lm5:.3f}')  # Muestra ambos valores para compararlos.
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

## 3.10. Transformar la respuesta: Box–Cox

### 3.10.1. Explorar una potencia

La dispersión y las colas de los residuos sugieren estudiar una transformación de `Strength`. Box–Cox requiere **valores estrictamente positivos**, condición que cumple esta respuesta:

$$
g_\lambda(y)=\begin{cases}(y^\lambda-1)/\lambda,&\lambda\ne0,\\ \log(y),&\lambda=0.\end{cases}
$$

$\lambda=0$ corresponde al logaritmo; $\lambda=1$ conserva la escala salvo un desplazamiento, irrelevante para un modelo con intercepto.

Para cada potencia, `logverosimilitud_boxcox` vuelve a estimar el modelo con **las mismas columnas** y evalúa la log-verosimilitud incluyendo el cambio de escala. Buscamos el máximo y dibujamos el perfil. Los valores plausibles al 95 % cumplen aproximadamente

$$
\ell(\lambda)\ge\ell(\widehat\lambda)-\frac{\chi^2_{0{,}95;1}}{2}.
$$

**Interpretación:** el perfil ayuda a escoger una potencia bajo este modelo; no demuestra mejora predictiva. El AIC ordinario de una regresión con respuesta transformada no se compara directamente con el de `Strength` sin tener en cuenta la transformación.

In [ ]:
from s3_logverosimilitud_boxcox import logverosimilitud_boxcox  # Importa el cálculo del perfil con cambio de escala.

optimo_lambda = minimize_scalar(lambda valor: -logverosimilitud_boxcox(valor, mod_lm5),  # Minimiza el negativo de la log-verosimilitud.
                                bracket=(-1, 1), method="brent")  # Usa un intervalo inicial de búsqueda.
if not optimo_lambda.success:  # Comprueba que la optimización terminó correctamente.
    raise RuntimeError("No se ha podido estimar lambda.")  # Detiene el análisis si no hubo convergencia.
lamb = optimo_lambda.x  # Guarda la potencia estimada solo con entrenamiento.

bc = pd.DataFrame({"lambda": np.linspace(-2, 2, 201)})  # Construye una rejilla de potencias para visualizar el perfil.
bc["logLik"] = [logverosimilitud_boxcox(valor, mod_lm5) for valor in bc["lambda"]]  # Evalúa el mismo modelo en cada potencia.
print(f'Lambda estimada: {lamb:.4f}')  # Presenta el máximo para interpretarlo.

In [ ]:
def explorar_boxcox(valor_lambda):  # Compara una potencia candidata con el perfil aprendido.
    fig, ax = plt.subplots(figsize=(16, 8))  # Crea el gráfico.
    ax.plot(bc["lambda"], bc["logLik"], color="#34699a", linewidth=2)  # Dibuja la log-verosimilitud perfilada.
    ax.axvline(lamb, color="#800000", linestyle="--", linewidth=1.8,  # Marca el máximo estimado.
               label=f"Lambda estimada: {lamb:.3f}")
    ax.axvline(valor_lambda, color="#6f9fc8", linewidth=1.8,  # Marca la potencia que queremos examinar.
               label=f"Lambda seleccionada: {valor_lambda:.2f}")
    corte = logverosimilitud_boxcox(lamb, mod_lm5) - stats.chi2.ppf(0.95, 1) / 2  # Calcula el corte aproximado del 95 %.
    ax.axhline(corte, color="#666666", linestyle=":", linewidth=1.5,  # Distingue potencias plausibles bajo el modelo.
               label="Corte del intervalo del 95 %")
    ax.set(xlabel="Lambda", ylabel="Log-verosimilitud", title="Perfil Box–Cox")  # Identifica ejes.
    ax.legend(fontsize=12)  # Explica las líneas.
    fig.tight_layout()  # Ajusta márgenes.
    plt.show()  # Muestra el perfil y la candidata.

In [ ]:
explorar_boxcox(1.0)  # Comprueba si mantener la escala original es plausible según el perfil.

### Lectura del perfil Box–Cox

El máximo se sitúa en **$\widehat\lambda\approx0{,}657$**. En el perfil, $\lambda=1$ queda por debajo del corte aproximado del 95 %: mantener la escala original resulta menos compatible con el criterio de verosimilitud de este ajuste.

**Qué no demuestra:** ni garantiza residuos perfectamente normales, ni elimina por sí solo todos los problemas de dispersión, ni prueba una mejora en test. El resultado está condicionado por las columnas seleccionadas y los supuestos del modelo.

#### Micropráctica: comprobar el logaritmo

Reutiliza `explorar_boxcox(1.0)` marcando ahora **$\lambda=0$**. Completa el hueco y comprueba si esa potencia queda por encima del corte. Explica la diferencia entre «una transformación posible» y «una potencia plausible según este perfil».

In [ ]:
try:
    lambda_prueba = ...  # Completa con lambda = 0, que corresponde a la transformación logarítmica.
    explorar_boxcox(lambda_prueba)  # Dibuja el perfil con el valor candidato marcado.
except Exception as error:
    plt.close("all")  # Evita dejar un gráfico parcial si el valor no es válido.
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### 3.10.2. Ajustar con la respuesta transformada

Creamos `Strength2` con $\log(y)$ si la potencia es prácticamente cero y con $y^\lambda$ en otro caso. Para una potencia fija no nula, esta última forma y Box–Cox estándar difieren en un cambio afín; el perfil sirve para elegir la potencia, pero el código indica la escala exacta del ajuste.

`mod_lm6` utiliza las **mismas siete variables polinómicas que `mod_lm5`**. `CoarseAggregate` ya había sido retirado por AIC; no estamos atribuyendo su retirada a Box–Cox.

**Cómo leer el resultado:** coeficientes y residuos están en la escala de `Strength2`, no en MPa. No compares su AIC ordinario ni el tamaño de sus residuos directamente con los de `mod_lm5`. Esta rama es exploratoria y no sustituirá al modelo final en escala original.

In [ ]:
def transformar_respuesta(valores, valor_lambda):  # Aplica la escala concreta utilizada en este ajuste ilustrativo.
    return np.log(valores) if abs(valor_lambda) < 1e-8 else np.power(valores, valor_lambda)  # Logaritmo o potencia.

datos["Strength2"] = transformar_respuesta(datos["Strength"], lamb)  # Añade la respuesta transformada sin sustituir Strength.
variables_sin_coarse = [variable for variable in predictores if variable != "CoarseAggregate"]  # Conserva el listado auxiliar.
columnas_sin_coarse = mod_lm5.model.exog_names  # Reutiliza exactamente las columnas seleccionadas; aquí no incluyen CoarseAggregate.
mod_lm6 = sm.OLS(datos["Strength2"], X_polinomico[columnas_sin_coarse], missing="raise").fit()  # Ajusta sobre las mismas filas.
display(mod_lm6.summary())  # Lee este resumen en la escala transformada.

### 3.10.3. Diagnosticar la respuesta transformada

Repetimos los cuatro paneles para `mod_lm6`. Compara con `mod_lm5` la **forma** de la curva residual, la uniformidad de la dispersión y las colas del Q–Q.

**No compares directamente la amplitud de los ejes:** ahora los residuos son `Strength2` observada menos ajustada, no resistencias en MPa. La cifra de $R^2$ también resume variación de otra respuesta.

La transformación puede mejorar un aspecto sin resolver los demás. Redacta qué parece mejorar y qué sigue requiriendo cautela.

In [ ]:
diagnosticos(mod_lm6, "Diagnóstico del modelo con Strength2")  # Examina los patrones en la nueva escala de respuesta.

### Lectura de los diagnósticos transformados

El Q–Q se acerca más a la recta y la curva de escala y localización se aplana en la parte alta. Sin embargo, la curva de residuos frente a ajustados sigue mostrando estructura, y permanecen casos con apalancamiento elevado.

**Conclusión:** la transformación mejora algunos patrones, pero no convierte automáticamente el ajuste en un modelo válido para cualquier uso. Las amplitudes residuales ahora corresponden a otra escala.

El ejercicio siguiente cambia **una predictora**, no la respuesta: así se podrán comparar AIC sobre `Strength` y las mismas filas.

## Ejercicio 3.1 — Transformar predictores con cuidado

Este ejercicio modifica **una predictora**, no `Strength`. Trabaja sobre una copia y conserva `datos`, `X_lineal` y los modelos originales; el análisis principal los reutiliza.

Resuelve los tres apartados en orden: **comprobar el dominio → ajustar una alternativa → diagnosticar y concluir**. En cada paso, reutiliza las operaciones de los ejemplos y redacta una conclusión, no solo una salida numérica.

La comparación de AIC será pertinente porque mantendremos la misma respuesta y las mismas filas. No sustituyas `mod_lm5` ni consultes test para elegir la alternativa.

### 3.1.1. Comprobar el dominio

Consulta los mínimos de las predictoras originales e identifica las columnas con ceros. En estos cálculos, $\log(x)$ requiere $x>0$; no apliques `np.log` a toda la tabla sin comprobarlo.

Verifica que `Cement` es positiva en todas las filas del entrenamiento y justifica por qué puede transformarse directamente. Los ceros de otros componentes no son valores ausentes ni deben eliminarse automáticamente.

In [ ]:
# 3.1.1 — Comprobar el dominio de las transformaciones.
# Pista: consulta los mínimos de predictores; un valor cero impide usar np.log directamente.
# Tu código:

### 3.1.2. Ajustar con el logaritmo del cemento

Parte de `X_lineal.copy()` y sustituye **solo** la columna `Cement` de la copia por su logaritmo. Ajusta un nuevo modelo con `sm.OLS(y, ...)` y guárdalo con otro nombre.

Compara su AIC con `mod_lm1.aic` y explica el sentido de la diferencia. Mantén las columnas de ruido, el intercepto y las mismas filas: cambiar varias cosas a la vez impediría atribuir el cambio de ajuste a esta transformación.

In [ ]:
# 3.1.2 — Ajustar un modelo con el logaritmo de Cement.
# Pista: parte de X_lineal.copy(), modifica solo la copia y ajusta con sm.OLS(y, ...).
# Compara su AIC con mod_lm1.aic; no sobrescribas los modelos originales.
# Tu código:

### 3.1.3. Diagnosticar y concluir

Aplica `diagnosticos(...)` al nuevo modelo y compara sus patrones con los del lineal. Para una comparación directa, puedes llamar también a `diagnosticos(mod_lm1, ...)`.

Redacta una conclusión con: **transformación aplicada, AIC comparado, cambios de curvatura y dispersión, colas e influencia**. Un AIC menor no demuestra que los supuestos sean correctos. Conserva el ajuste principal y reserva el test para la evaluación final.

In [ ]:
# 3.1.3 — Diagnosticar el modelo alternativo.
# Pista: reutiliza diagnosticos(...) y contrasta los patrones con el modelo sin logaritmos.
# Tu código:

## 3.11. Observaciones influyentes

### 3.11.1. Tres conceptos diferentes

| Medida | Qué describe |
|---|---|
| Residuo studentizado externo | Cuánto se aleja la respuesta, teniendo en cuenta la escala estimada sin ese caso |
| Apalancamiento | Cuán inusual es su combinación de predictoras |
| Distancia de Cook | Cuánto puede cambiar el ajuste al retirar la observación |

Un caso puede destacar en una medida y no en otra. Calculamos las tres para `mod_lm5`, con todas las filas originales.

Al examinar muchos residuos, ajustamos sus p-valores con **Bonferroni**, una corrección conservadora. Los gráficos mostrarán $-\log_{10}(p_{ajustado})$: por encima de la referencia hay un p ajustado menor que 0,05.

**Qué hacer con una señal:** localizar el caso, revisar sus variables originales y estudiar sensibilidad. Ninguna medida confirma por sí sola que el dato sea erróneo ni justifica eliminarlo automáticamente.

In [ ]:
with threadpool_limits(limits=1):  # Limita los hilos durante los reajustes de residuos externos.
    influencia_modelo = mod_lm5.get_influence()  # Obtiene las medidas del modelo polinómico entrenado.
    studentizados = influencia_modelo.resid_studentized_external  # Estima cada residuo usando la escala sin ese caso.

influencia = pd.DataFrame({  # Reúne las medidas con las etiquetas originales de observación.
    "Cook": influencia_modelo.cooks_distance[0],  # Mide la influencia sobre el ajuste.
    "Residuo studentizado": studentizados,  # Permite comparar residuos de distintas observaciones.
    "p Bonferroni": np.minimum(1, 2 * stats.t.sf(np.abs(studentizados), mod_lm5.df_resid - 1) * len(datos)),  # Ajusta p-valores bilaterales.
    "Apalancamiento": influencia_modelo.hat_matrix_diag,  # Mide combinaciones inusuales de entradas.
}, index=datos.index)
display(influencia.loc[[65, 81, 147, 248]])  # Muestra cuatro casos que compararemos en los gráficos.

### Lectura de los casos señalados

- **65 y 81:** tienen apalancamientos cercanos a **0,217**, pero residuos diferentes; la distancia de Cook es mayor para 81.
- **147:** combina un residuo studentizado cercano a **−3,04** con apalancamiento elevado y Cook apreciable.
- **248:** tiene un residuo mayor en magnitud (**3,87**), pero mucho menos apalancamiento. Su p-valor ajustado es aproximadamente **0,084**, no inferior a 0,05.

**Implicación:** el mayor residuo no tiene por qué coincidir con la mayor influencia. La corrección por muchas comparaciones también cambia qué casos se consideran extremos por ese contraste.

### 3.11.2. Indicadores por observación

Dibujamos cada medida frente a la etiqueta del caso. En el panel de p-valores, los valores ajustados iguales a 1 quedan en cero; la línea de referencia corresponde al 5 %. Las etiquetas facilitan recuperar los casos, no indican que deban borrarse.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 11))  # Reserva un panel para cada indicador.
for ax, columna in zip(axes.flat, influencia.columns):  # Recorre las cuatro medidas.
    if columna == "p Bonferroni":  # Representa la significación en una escala legible.
        valores = -np.log10(influencia[columna].clip(lower=np.finfo(float).tiny))  # Evita logaritmos de cero.
        significativos = influencia[columna] < 0.05  # Identifica p-valores ajustados menores que alfa.
        _ = ax.scatter(influencia.index[~significativos], valores[~significativos],  # Dibuja el resto de casos.
                       s=30, color="#9fc3e4", edgecolor="#666666", linewidth=0.4, alpha=0.65)
        if significativos.any():  # Destaca solo si existen casos significativos tras el ajuste.
            _ = ax.scatter(influencia.index[significativos], valores[significativos],
                           s=60, color="#800000", edgecolor="white", linewidth=0.6,
                           label="p ajustado < 0,05")
        _ = ax.axhline(-np.log10(0.05), color="#800000", linestyle="--", linewidth=1.8,  # Marca el umbral del 5 %.
                       label="Umbral: p ajustado = 0,05")
        _ = ax.set_ylabel("−log₁₀(p Bonferroni)")  # Identifica la transformación del eje.
        _ = ax.legend(fontsize=10)  # Explica las referencias.
    else:  # Conserva las otras medidas en su escala propia.
        _ = ax.vlines(influencia.index, 0, influencia[columna], color="#6f9fc8", alpha=0.7)  # Marca cada indicador.
        valores = influencia[columna]  # Guarda las coordenadas para anotar casos.
        _ = ax.set_ylabel(columna)  # Identifica la medida.
    _ = ax.set(xlabel="Observación", title=columna)  # El eje horizontal usa etiquetas, no fechas.
    for observacion in [65, 81, 147, 248]:  # Señala los casos de la tabla anterior.
        if columna == "p Bonferroni" and influencia.loc[observacion, columna] == 1:
            continue  # Evita amontonar etiquetas de p-valores iguales a 1 en cero.
        desplazamiento = (-14, 8) if columna == "Apalancamiento" and observacion == 65 else (4, 5)  # Separa etiquetas.
        alineacion = "right" if desplazamiento[0] < 0 else "left"  # Ajusta su alineación.
        _ = ax.annotate(str(observacion), (observacion, valores.loc[observacion]),  # Localiza el caso por su etiqueta.
                        xytext=desplazamiento, textcoords="offset points", fontsize=10,
                        ha=alineacion, color="#800000")
    ax.title.set_fontsize(18)  # Ajusta los textos de cada panel.
    ax.xaxis.label.set_fontsize(16)
    ax.yaxis.label.set_fontsize(16)
    ax.tick_params(labelsize=14)
fig.tight_layout()  # Evita solapamientos.
plt.show()  # Muestra los indicadores por observación.

### Lectura de los indicadores

Cook destaca especialmente el caso 81, mientras el residuo studentizado destaca el 248. Ningún punto del panel de Bonferroni supera la referencia del 5 %; eso no demuestra que todos los datos sean correctos ni elimina la necesidad de revisar sensibilidad.

#### Micropráctica: localizar los casos más influyentes

Aplica el patrón `nlargest(3)` a `influencia['Cook']`. Completa el número de casos y compara las etiquetas con las de mayor residuo absoluto.

Explica por qué ambas listas pueden diferir. No borres filas: en el siguiente apartado examinaremos las combinaciones de residuo y apalancamiento.

In [ ]:
try:
    # Lista las tres observaciones con mayor distancia de Cook.
    cook_mayores = influencia['Cook'].nlargest(...)  # Completa el número de casos que quieres inspeccionar.
    print(cook_mayores)  # Muestra los índices y valores de Cook, de mayor a menor.
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### 3.11.3. Combinar residuo y apalancamiento

Cada burbuja representa una observación: **apalancamiento en el eje horizontal**, **residuo studentizado externo en el vertical** y tamaño proporcional a Cook respecto al máximo de esta muestra.

Las referencias horizontales en $\pm2$ y verticales en $2p/n$ y $3p/n$ ayudan a localizar casos; $p$ cuenta los parámetros, incluido el intercepto. **Son referencias heurísticas, no pruebas de error de medición.**

Compara 65 con 81 y después con 248. ¿Qué combinación explica que una burbuja sea grande? Las etiquetas se pueden mostrar u ocultar cambiando `True` por `False`.

In [ ]:
def grafico_influencia(etiquetas):  # Relaciona las tres medidas en un mismo gráfico.
    fig, ax = plt.subplots(figsize=(16, 8))  # Crea la figura.
    tamanos = 20 + 800 * influencia["Cook"] / influencia["Cook"].max()  # Escala burbujas respecto al máximo de Cook.
    ax.scatter(influencia["Apalancamiento"], influencia["Residuo studentizado"],  # Sitúa cada observación.
               s=tamanos, color="#9fc3e4", alpha=0.65,
               edgecolors="#666666", linewidth=0.6)  # Hace visibles los bordes.
    ax.axhline(0, color="#800000", linestyle="--", linewidth=1.8)  # Marca residuo cero.
    for valor in [-2, 2]:  # Añade referencias de magnitud residual.
        ax.axhline(valor, color="#666666", linestyle=":", linewidth=1.2)
    for factor in [2, 3]:  # Añade referencias heurísticas de apalancamiento.
        ax.axvline(factor * len(mod_lm5.params) / len(datos), color="#800000",
                   linestyle=":", linewidth=1.5)  # Cuenta también el intercepto entre los parámetros.
    if etiquetas:  # Permite mostrar u ocultar los casos seleccionados.
        for observacion in [65, 81, 147, 248]:
            fila = influencia.loc[observacion]  # Recupera las medidas por etiqueta.
            ax.annotate(str(observacion), (fila["Apalancamiento"], fila["Residuo studentizado"]),
                        xytext=(6, 5), textcoords="offset points", fontsize=12,
                        color="#800000")  # Identifica el caso para revisar sus datos.
    ax.set(xlabel="Apalancamiento", ylabel="Residuo studentizado",
           title="Influencia de las observaciones")  # Identifica los ejes.
    fig.tight_layout()  # Ajusta márgenes.
    plt.show()  # Muestra la combinación de indicadores.

In [ ]:
grafico_influencia(True)  # Muestra las etiquetas de los cuatro casos de referencia; False las oculta.

### 3.11.4. Inspeccionar las observaciones

Recuperamos los casos **81, 147 y 248** por sus etiquetas. Revisa las entradas originales y `Strength`: ¿destaca una cantidad, una edad o una combinación de componentes?

`obs_rm` es solo una lista para el análisis de sensibilidad posterior: **todavía no se eliminan filas**. No es necesariamente la lista de los tres mayores valores de Cook. El caso 65 sigue siendo útil para comparar apalancamiento, aunque no forme parte de esta selección ilustrativa.

Un punto destacado en un gráfico no demuestra una errata; comprueba primero la tabla y el contexto de medición.

In [ ]:
obs_rm = [81, 147, 248]  # Guarda los casos que revisaremos, sin borrarlos de datos.
display(datos.loc[obs_rm, columnas_originales])  # Muestra composición, edad y resistencia originales, sin Strength2.

### Lectura de los casos originales

El caso 81 tiene **32,2** de `Superplasticizer`, el máximo observado; el 147 tiene **121,8** de `Water`, el mínimo de entrenamiento. El 248 tiene una resistencia de **78,80 MPa**, alta dentro del rango observado. Estos datos ayudan a entender las señales, pero no demuestran errores de medición.

### 3.11.5. Localizar los casos en las relaciones

Repetimos la matriz de dispersión destacando 81, 147 y 248 con colores y etiquetas. Así se puede ver si un caso es inusual en una variable o en una combinación.

**Cautela:** parecer ordinario en cada panel bivariante no descarta apalancamiento multivariable. El color solo sirve para encontrar el caso y compararlo con la tabla anterior.

In [ ]:
col_points = np.array(["#9fc3e433"] * len(datos), dtype=object)  # Deja el resto de observaciones en segundo plano.
tamanos = np.full(len(datos), 16.0)  # Asigna un tamaño inicial común.
colores_destacados = ["#800000", "#34699a", "#6f9fc8"]  # Usa un color por caso seleccionado.
for observacion, color in zip(obs_rm, colores_destacados):  # Recorre las etiquetas originales.
    posicion = datos.index.get_loc(observacion)  # Convierte la etiqueta en posición para los arreglos gráficos.
    col_points[posicion] = color  # Destaca el caso sin modificar sus valores.
    tamanos[posicion] = 85  # Aumenta su visibilidad.

axes = scatter_matrix(datos[columnas_originales], figsize=(18, 18), diagonal="hist",  # Mantiene solo las nueve variables originales.
                      c=col_points, s=tamanos, alpha=None,  # Respeta la transparencia de cada color.
                      hist_kwds={"color": "#9fc3e4", "edgecolor": "#666666"})
for i, variable_y in enumerate(columnas_originales):  # La fila determina el eje vertical.
    for j, variable_x in enumerate(columnas_originales):  # La columna determina el horizontal.
        eje = axes[i, j]  # Recupera el panel correspondiente.
        eje.tick_params(labelsize=9)  # Ajusta la legibilidad.
        eje.xaxis.label.set_fontsize(11)
        eje.yaxis.label.set_fontsize(11)
        if i != j:  # Añade etiquetas solo en gráficos de pares, no en histogramas.
            for observacion, color in zip(obs_rm, colores_destacados):
                _ = eje.annotate(str(observacion),
                                 (datos.loc[observacion, variable_x], datos.loc[observacion, variable_y]),  # Usa valores originales.
                                 xytext=(3, 3), textcoords="offset points", fontsize=8, color=color)
plt.show()  # Muestra dónde están los casos en las relaciones bivariantes.

### 3.11.6. Reajustar como análisis de sensibilidad

Creamos una copia `datos_rm` sin 81, 147 y 248 y ajustamos `mod_lm7`. Conservamos **las mismas columnas y los mismos centros y escalas de `mod_lm5`** para que la comparación de coeficientes no mezcle la retirada de filas con un cambio de base.

La muestra reducida tiene 703 observaciones. Los datos y el modelo originales siguen intactos. Comparamos cuánto cambian los coeficientes; este ejercicio no prueba que los tres casos deban eliminarse.

**Cautela:** no compares directamente AIC de ajustes sobre 706 y 703 filas: sus muestras son distintas. Una reducción del error al quitar casos difíciles tampoco demuestra mejor generalización. El modelo final conservará todas las observaciones de entrenamiento.

In [ ]:
datos_rm = datos.drop(index=obs_rm).copy()  # Excluye tres casos en una copia; datos permanece intacto.
X_rm, _ = diseno_polinomico(datos_rm, parametros_polinomicos)  # Reutiliza la base original para comparar coeficientes.
mod_lm7 = sm.OLS(datos_rm["Strength"], X_rm[mod_lm5.model.exog_names], missing="raise").fit()  # Conserva la especificación seleccionada.
display(mod_lm7.summary())  # Examina el reajuste sobre las 703 filas restantes.

In [ ]:
comparacion_coeficientes = pd.DataFrame({  # Compara coeficientes con las mismas columnas y escalas.
    "Con todos los casos": mod_lm5.params,  # Ajuste principal sobre 706 filas.
    "Sin los tres casos": mod_lm7.params,  # Ajuste de sensibilidad sobre 703 filas.
})
comparacion_coeficientes["Cambio"] = mod_lm7.params - mod_lm5.params  # Calcula cambio: reducido menos original.
display(comparacion_coeficientes)  # Examina estabilidad; no es una prueba de que deban excluirse casos.

### Lectura del análisis de sensibilidad

Los coeficientes cambian en distinta medida: el término lineal de `Cement` apenas cambia, mientras los de `Water` y `Superplasticizer` se desplazan más. Esta comparación informa sobre sensibilidad a los tres casos, no sobre qué muestra representa mejor la población.

## 3.12. Modelo final e interpretación

### 3.12.1. Fijar el modelo antes de consultar test

Usaremos **`mod_final = mod_lm5`**: el polinómico seleccionado con todas las observaciones de entrenamiento y respuesta `Strength` en escala original. Los ajustes transformado y reducido son análisis complementarios.

**Regla:** a partir de aquí el test servirá para evaluar, no para volver atrás y elegir otra especificación. Cambiar el modelo por sus resultados convertiría el test en parte del entrenamiento.

In [ ]:
mod_final = mod_lm5  # Fija el modelo entrenado sobre todas las filas y en MPa, antes de evaluar test.

### 3.12.2. Visualizar la relación ajustada de cada predictor

Variamos una entrada dentro de su rango de entrenamiento y predecimos la resistencia media. Para un término cuadrático, su pendiente cambia con el valor de la entrada:

$$
\frac{\partial\widehat y}{\partial x_j}=\frac{\widehat\beta_{j1}+2\widehat\beta_{j2}z_j}{s_{j,\mathrm{train}}}.
$$

La función crea 100 valores, reconstruye sus columnas lineal y cuadrática y deja las demás **columnas del diseño** en sus medias de entrenamiento. La línea es la respuesta media estimada y la banda su **IC del 95 % bajo el modelo**, no el intervalo de una pieza individual.

**Cautelas:** la media de una columna cuadrática no es el cuadrado de la media de su entrada. Este perfil de diseño puede no representar una receta real; tampoco es un efecto causal. Los intervalos dependen de los supuestos y no incorporan toda la incertidumbre de la selección. No extrapoles fuera de los rangos mostrados.

In [ ]:
def datos_efecto(variable):  # Construye un perfil para una entrada seleccionada.
    valores = np.linspace(datos[variable].min(), datos[variable].max(), 100)  # Usa su rango de entrenamiento.
    columnas = mod_final.model.exog_names  # Conserva el orden esperado por el modelo.
    medias = mod_final.model.exog.mean(axis=0)  # Fija las otras columnas en medias del diseño.
    matriz = pd.DataFrame(np.tile(medias, (len(valores), 1)), columns=columnas)  # Crea una fila por valor explorado.
    if variable in parametros_polinomicos:  # Reconstruye juntos los dos términos cuadráticos.
        centro, escala = parametros_polinomicos[variable]  # Usa la estandarización aprendida.
        z = (valores - centro) / escala  # Calcula el término lineal estandarizado.
        matriz[grupos_polinomicos[variable]] = np.column_stack((z, z ** 2))  # Mantiene coherentes ambas columnas.
    else:
        matriz[variable] = valores  # Para un término lineal, sustituye solo su columna.
    resumen = mod_final.get_prediction(matriz).summary_frame(alpha=0.05)  # Calcula media e intervalos al 95 %.
    return valores, resumen  # Devuelve coordenadas y resultados del modelo.

def dibujar_efecto(ax, variable, banda=True):  # Dibuja el perfil en un eje ya creado.
    valores, resumen = datos_efecto(variable)  # Obtiene las predicciones del escenario.
    if banda:  # Permite mostrar u ocultar el IC de la media.
        ax.fill_between(valores, resumen["mean_ci_lower"].to_numpy(), resumen["mean_ci_upper"].to_numpy(),
                        color="#9fc3e4", alpha=0.5, label="IC de la media (95 %)")  # No es un IP individual.
    ax.plot(valores, resumen["mean"].to_numpy(), color="#800000", linewidth=1.8)  # Representa la respuesta media estimada.
    ax.set(xlabel=variable, ylabel="Strength media predicha (MPa)", title=variable)  # Identifica respuesta y unidades.

In [ ]:
filas_efectos = int(np.ceil(len(seleccion_polinomica) / 4))  # Calcula cuántas filas de paneles hacen falta.
fig, axes = plt.subplots(filas_efectos, 4, figsize=(20, 5 * filas_efectos), squeeze=False)  # Reserva la cuadrícula.
for ax, variable in zip(axes.flat, seleccion_polinomica):  # Recorre solo entradas del modelo final.
    dibujar_efecto(ax, variable)  # Dibuja cada media predicha con su IC.
    ax.title.set_fontsize(16)  # Ajusta textos para comparar los paneles.
    ax.xaxis.label.set_fontsize(13)
    ax.yaxis.label.set_fontsize(13)
    ax.tick_params(labelsize=11)
for ax in list(axes.flat)[len(seleccion_polinomica):]:  # Localiza paneles vacíos.
    ax.set_visible(False)  # Oculta los sobrantes.
fig.tight_layout()  # Evita solapamientos.
plt.show()  # Muestra los perfiles del modelo seleccionado.

### Lectura de los perfiles ajustados

`Cement` presenta una relación creciente casi recta, mientras otros términos muestran curvatura. En `Age`, el perfil cuadrático aumenta y luego disminuye: es la forma de **este modelo y este escenario**, no una prueba causal de pérdida de resistencia.

Las bandas suelen ensancharse hacia extremos menos apoyados por datos. Se refieren a la **media estimada**, no al 95 % de piezas individuales. Los perfiles pueden diferir de las curvas marginales porque ahora se usa el modelo conjunto.

### 3.12.3. Examinar un predictor con más detalle

La función siguiente reutiliza el mismo perfil. Puedes elegir una entrada de `seleccion_polinomica` y mostrar u ocultar la banda sin reajustar el modelo.

In [ ]:
def explorar_efecto(variable, banda):  # Examina una entrada seleccionada sin modificar el ajuste.
    fig, ax = plt.subplots(figsize=(16, 8))  # Prepara una figura más amplia.
    dibujar_efecto(ax, variable, banda)  # Reutiliza la curva y, si se pide, el IC de la media.
    if banda:
        ax.legend(fontsize=12)  # Identifica la banda de confianza.
    fig.tight_layout()  # Ajusta márgenes.
    plt.show()  # Muestra el perfil elegido.

In [ ]:
explorar_efecto(seleccion_polinomica[0], True)  # Muestra el primer predictor seleccionado con el IC de su respuesta media.

#### Micropráctica: curva e incertidumbre

Usa `explorar_efecto` con `Age` y oculta la banda pasando `False` como segundo argumento. Completa el hueco y anticipa qué cambiará: ¿la curva, la banda o ambas?

Compara con el panel anterior y explica por qué ocultar un intervalo **no elimina la incertidumbre** ni vuelve a estimar el modelo.

In [ ]:
try:
    mostrar_banda = ...  # Completa con False para ocultar el intervalo de confianza.
    if not isinstance(mostrar_banda, bool):  # Detecta el hueco sin completar antes de dibujar.
        raise ValueError("Indica True o False para la banda.")
    explorar_efecto('Age', mostrar_banda)  # Reutiliza el perfil del mismo modelo entrenado.
    # Explica qué cambia en la figura y qué permanece igual en el modelo.
except Exception as error:
    plt.close("all")  # Cierra una posible figura incompleta.
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

## 3.13. Evaluar con datos nuevos

### 3.13.1. Leer el conjunto de test

Hasta ahora ajustamos, seleccionamos y diagnosticamos **solo con entrenamiento**. Leemos ahora casos reservados para evaluar el modelo final, usando el mismo separador tabulado y numerando sus filas para seleccionar observaciones por etiqueta.

La columna auxiliar `Strength2` utiliza la potencia aprendida en entrenamiento; **no participa en `mod_final`**, que predice `Strength` en MPa. No se recalculan la potencia, las medias ni las escalas con test.

**Regla:** sus respuestas observadas se usarán para medir errores y representar resultados, no para reajustar o elegir el modelo.

In [ ]:
test = pd.read_csv(DATA_PATH / "Concrete_test.txt", sep="\t", na_values=["NA", ""])  # Lee los casos reservados para evaluación.
test.index = np.arange(1, len(test) + 1)  # Asigna etiquetas propias al conjunto de test.
test.index.name = "observacion"  # Permite seleccionar cada caso con loc.
test["Strength2"] = transformar_respuesta(test["Strength"], lamb)  # Mantiene la transformación auxiliar; mod_final no la utiliza.
print(f'Observaciones de test: {len(test)}')  # Comprueba el tamaño de la muestra de evaluación.
display(test[columnas_originales].head())  # Revisa las primeras filas sin mezclar la columna auxiliar.

La evaluación dispone de **324 observaciones de test**, distintas de las 706 filas de entrenamiento. El número de test no es el número de parámetros ni una ampliación automática de la muestra de ajuste.

### 3.13.2. Construir el diseño y predecir

`diseno_polinomico(test, parametros_polinomicos)` aplica las **medias y escalas de entrenamiento** a las entradas nuevas. Seleccionamos `mod_final.model.exog_names` para respetar exactamente las columnas y su orden.

`predict` produce una resistencia estimada por caso; `pr` conserva el índice para emparejarla después con `test['Strength']`. Ni `Strength` ni `Strength2` se usan como entradas del modelo.

**Comprobación:** debe haber una predicción por observación, sin desalineación de filas. Si aparecen valores extraños, revisa dominio y preprocesamiento antes de atribuirlos al algoritmo.

In [ ]:
X_test, _ = diseno_polinomico(test, parametros_polinomicos)  # Reutiliza el preprocesamiento aprendido, sin volver a estimarlo.
pr = mod_final.predict(X_test[mod_final.model.exog_names]).rename("Prediccion")  # Predice con las columnas en el orden correcto.
print(f'Predicciones obtenidas: {len(pr)}')  # Comprueba que hay una por cada fila de test.
display(pr.head())  # Muestra los primeros resultados y conserva pr completa para la evaluación.

### 3.13.3. Comparar predicción y observación

Cada punto representa un caso de test: **predicción en el eje horizontal** y **resistencia observada en el vertical**. La diagonal indica igualdad; por encima, el modelo subestima, y por debajo, sobreestima. La tabla muestra entradas originales, predicción y residuo **observado menos predicho**.

En el caso seleccionado se muestran dos intervalos al 95 % calculados con el modelo entrenado:

| Intervalo | Qué estima |
|---|---|
| IC de la media | Respuesta media para esas entradas |
| IP individual | Respuesta de una nueva observación con esas entradas |

El IP suele ser más ancho porque añade la variabilidad individual. **Ninguno se calcula usando la resistencia real del caso mostrado**; esa respuesta solo permite comparar la predicción con lo observado. Los intervalos dependen de los supuestos del modelo y no cubren toda la incertidumbre de selección.

Desliza el control o escribe la etiqueta del caso. Si el widget no se representa, muestra un caso con `display(explorar_prediccion(5))`. Un único punto fuera del IP no decide por sí solo la validez del modelo.

In [ ]:
from functools import partial  # Permite fijar argumentos que no cambian entre casos.
from s3_explorar_prediccion import explorar_prediccion as _explorar_prediccion  # Importa la vista de gráfico y tabla.

explorar_prediccion = partial(  # Deja la etiqueta de observación como único argumento de la exploración.
    _explorar_prediccion, mod_final, X_test, pr, test, columnas_originales  # Fija el modelo y los datos ya preparados.
)

In [ ]:
from ipywidgets import BoundedIntText, HBox, HTML, IntSlider, Layout, VBox, link  # Importa controles y composición visual.

selector_predicciones = IntSlider(  # Crea un deslizador con etiquetas válidas de test.
    value=5, min=int(test.index.min()), max=int(test.index.max()), step=1,
    description="Observación", continuous_update=False, readout=False,  # Actualiza al terminar el desplazamiento.
    style={"description_width": "105px"}, layout=Layout(width="520px"),
)
numero_predicciones = BoundedIntText(  # Permite escribir el caso sin salir del rango de test.
    value=5, min=selector_predicciones.min, max=selector_predicciones.max,
    description="Ir al n.º", style={"description_width": "65px"},
    layout=Layout(width="170px"),
)
enlace_predicciones = link((selector_predicciones, "value"), (numero_predicciones, "value"))  # Sincroniza los dos controles.
visor_predicciones = HTML(value=explorar_prediccion(selector_predicciones.value).data,  # Muestra el primer caso.
                          layout=Layout(width="100%"))

def actualizar_prediccion(cambio):  # Reacciona a una nueva etiqueta seleccionada.
    visor_predicciones.value = explorar_prediccion(cambio["new"]).data  # Cambia solo la vista; no reajusta el modelo.

selector_predicciones.observe(actualizar_prediccion, names="value")  # Conecta el cambio de caso con la actualización.
control_predicciones = VBox([  # Organiza título, controles y gráfico con tabla.
    HTML("<div style='border-left: 4px solid #800000; padding: 8px 12px; "
         "font-size: 18px; font-weight: 600;'>Explora las predicciones en test</div>"),
    HBox([selector_predicciones, numero_predicciones], layout=Layout(align_items="center")),
    visor_predicciones,
], layout=Layout(width="100%", gap="12px"))
display(control_predicciones)  # Muestra el explorador individual en el cuaderno.

### Lectura del caso de test 5

La predicción es aproximadamente **25,21 MPa** y la resistencia observada, **45,85 MPa**. El punto está por encima de la diagonal: el modelo subestima este caso, con residuo observado menos predicho de unos **20,64 MPa**.

Esto describe una observación concreta, no el error medio del test. Explora otros casos antes de resumir la calidad global, sin usar sus resultados para cambiar el ajuste.

#### Micropráctica: aumentar la confianza de los intervalos

Para el caso 5, usa `get_prediction(...).summary_frame(...)`, como en los perfiles anteriores, y cambia la confianza al **99 %**. Completa `alpha = 1 - confianza` y muestra las columnas del IC de la media y del IP individual.

Compara sus amplitudes con las del visor al 95 %. ¿Cuál incorpora la variabilidad de una observación individual? ¿Cambiará la predicción media al pedir más confianza?

In [ ]:
try:
    confianza_intervalos = ...  # Completa con la confianza solicitada, expresada entre 0 y 1.
    alpha_intervalos = 1 - confianza_intervalos  # Convierte confianza en nivel alfa.
    caso_intervalos = X_test.loc[[5], mod_final.model.exog_names]  # Usa únicamente las entradas de ese caso.
    resumen_intervalos = mod_final.get_prediction(caso_intervalos).summary_frame(alpha=alpha_intervalos)  # No usa su respuesta real.
    display(resumen_intervalos[['mean', 'mean_ci_lower', 'mean_ci_upper', 'obs_ci_lower', 'obs_ci_upper']])  # Media, IC e IP.
    # Compara anchuras con el visor y redacta qué incertidumbre resume cada intervalo.
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

## 3.14. Cuantificar el error en test

### 3.14.1. Error cuadrático medio y RMSE

Comparamos cada predicción con su respuesta observada **en las mismas filas**. Usaremos $\widehat y_i-y_i$; invertir el signo no cambia cuadrados ni valores absolutos:

$$
EQM=\frac{1}{n_{test}}\sum_i(\widehat y_i-y_i)^2,
\qquad RMSE=\sqrt{EQM}.
$$

El EQM penaliza especialmente los errores grandes y se expresa en **MPa²**. Su raíz, RMSE, vuelve a **MPa**, las unidades de `Strength`.

**Interpretación:** resumen del error en las 324 observaciones reservadas, no una precisión garantizada para cada futura pieza. No es el error de entrenamiento ni basta por sí solo para detectar sesgos sistemáticos.

In [ ]:
if not pr.index.equals(test.index):  # Comprueba que cada predicción corresponde a su fila de test.
    raise ValueError("Las predicciones y las observaciones de test no están alineadas.")
if pr.isna().any() or test["Strength"].isna().any():  # Evita reducir el error silenciosamente por valores ausentes.
    raise ValueError("Revisa los valores ausentes antes de evaluar las predicciones.")
n = test.shape[0]  # Cuenta las observaciones de la muestra de evaluación.
EQM = ((pr - test["Strength"]) ** 2).sum() / n  # Promedia los errores cuadrados en las mismas filas.
print(f'EQM en test: {EQM:.2f} MPa²')  # Muestra el error con sus unidades.

In [ ]:
RMSE = np.sqrt(EQM)  # Toma la raíz para volver de MPa² a MPa.
print(f'RMSE en test: {RMSE:.2f} MPa')  # Resume la magnitud de los errores dando más peso a los grandes.

### Lectura del error cuadrático

El EQM es aproximadamente **72,97 MPa²** y su raíz, **8,54 MPa**. El RMSE resume los errores de estas 324 predicciones; no significa que cada error sea de 8,54 MPa ni que todas las predicciones queden dentro de ese margen.

**Implicación:** conviene acompañarlo de una medida menos sensible a errores grandes y de la revisión de casos individuales. La tolerancia aceptable depende del uso del hormigón, no de la métrica por sí sola.

#### Micropráctica: error absoluto medio en test

El **MAE** es $\frac{1}{n_{test}}\sum_i|\widehat y_i-y_i|$. La celda siguiente muestra su cálculo con `abs()` y `mean()` entre `pr` y `test['Strength']`: identifica cada operación y exprésalo también en una sola línea.

¿En qué unidades se expresa? Compáralo con `RMSE`, que da más peso a los errores grandes, y explica por qué no tienen que coincidir.

In [ ]:
# Calcula el error absoluto medio de las predicciones sobre el conjunto de test.
error_absoluto = (pr - test['Strength']).abs()  # Calcula la magnitud del error fila por fila.
MAE = error_absoluto.mean()  # Promedia las magnitudes de los errores.
print(f'MAE en test: {MAE:.2f} MPa')  # Muestra el error en las mismas unidades que Strength.

### 3.14.2. Resumen de métricas en test

La tabla reúne **EQM, RMSE, MAE, MAPE y $R^2$ de test**. Calcula de nuevo el MAE a partir de los errores para que el resumen no dependa de haber resuelto una micropráctica; puedes compararlo con tu resultado anterior.

El **MAPE** promedia $100|\widehat y_i-y_i|/|y_i|$ y se expresa en porcentaje. No se calcula si alguna respuesta vale cero; cerca de cero puede resultar muy grande aunque el error en MPa sea moderado. Por eso no reemplaza a RMSE y MAE.

El **$R^2$ de test** es

$$
R^2_{test}=1-\frac{\sum_i(\widehat y_i-y_i)^2}{\sum_i(y_i-\bar y_{test})^2}.
$$

La media observada del test aparece **solo como referencia de evaluación**, no para entrenar el modelo. El resultado puede ser negativo si los errores del modelo superan esa referencia. Si todas las respuestas son iguales, el denominador es cero y se devuelve `NaN`.

**Regla:** estas métricas describen la evaluación del modelo ya fijado. No las uses para volver atrás y elegir otra especificación con el mismo test.

In [ ]:
observados = test["Strength"]  # Recupera las respuestas reservadas para evaluación.
errores = pr - observados  # Calcula predicción menos observación en las mismas filas.
MAE_resumen = errores.abs().mean()  # Recalcula el MAE sin depender de una variable del ejercicio.
MAPE = np.nan if (observados == 0).any() else (100 * errores.abs() / observados.abs()).mean()  # Evita dividir por cero.
variacion_test = ((observados - observados.mean()) ** 2).sum()  # Mide la variación total observada en test.
R2_test = np.nan if variacion_test == 0 else 1 - (errores ** 2).sum() / variacion_test  # Compara el error con esa referencia.
metricas_test = pd.Series({  # Reúne las métricas e identifica sus unidades.
    "EQM (MPa²)": EQM, "RMSE (MPa)": RMSE, "MAE (MPa)": MAE_resumen,
    "MAPE (%)": MAPE, "R² test": R2_test,
}, name="Evaluación en test")
display(metricas_test.to_frame())  # Muestra la tabla sin modificar el modelo ni sus predicciones.

### Lectura de la tabla de métricas

- **RMSE = 8,54 MPa y MAE = 6,46 MPa:** el RMSE es mayor porque penaliza más los errores grandes. Ninguno garantiza un margen fijo para todas las piezas.
- **MAPE ≈ 23,89 %:** resume errores relativos a la resistencia observada. Las resistencias pequeñas reciben mucho peso; no significa un 76,11 % de predicciones correctas.
- **$R^2_{test}\approx0{,}754$:** la suma de errores cuadrados es menor que la de la referencia basada en la media del test. No es el $R^2$ de entrenamiento ni una proporción de aciertos individuales.

**Implicación:** estas cifras describen el rendimiento en este conjunto reservado. Para juzgar su utilidad necesitamos las tolerancias del problema, la revisión de errores y saber qué población representan los datos.

### 3.14.3. Contextualizar la escala de la resistencia

Las dos celdas siguientes muestran la desviación típica y el resumen de `Strength` **observada en test**. Describen la variabilidad y el rango de las respuestas, no nuevos errores de predicción. No uses estos resultados para reajustar el modelo.

In [ ]:
test["Strength"].std(ddof=1)  # Describe la dispersión observada de las resistencias, en MPa; no es el RMSE.

In [ ]:
display(test["Strength"].describe())  # Resume las resistencias observadas de test: tamaño, centro, dispersión y extremos.

### Lectura de la escala de test

La resistencia observada tiene media **37,14 MPa**, mediana **36,08 MPa** y desviación típica **17,25 MPa**. Los valores van de **3,32 a 81,75 MPa**.

El RMSE de 8,54 MPa y el MAE de 6,46 MPa son menores que esa dispersión, pero compararlos no demuestra por sí solo que el modelo sea suficientemente preciso para un uso técnico. La desviación describe respuestas; las otras métricas describen **errores**.

**Siguiente paso:** comparar el modelo final con un referente sencillo ya entrenado, sin reajustar ninguno ni usar esta evaluación para elegir otra especificación.

## Ejercicio 3.2 — Comparar predicciones en test

Compara el modelo final con un **referente sencillo previamente entrenado**, `mod_lm0`. Usa las mismas filas de test y conserva los modelos y `pr` sin cambios.

Resuelve los dos apartados en orden: **construir el diseño y predecir → calcular RMSE y concluir**. Reutiliza las operaciones mostradas para el modelo final, pero adapta las columnas a las que conoce el modelo simple.

**Regla:** no ajustes ni selecciones modelos con este test. La comparación describe cuál predice mejor en esta muestra; no demuestra que esa ventaja se repita en toda futura población.

### 3.2.1. Predecir con el referente simple

Construye una matriz para test con **constante y `Cement`**, como `X_simple`, y selecciona `mod_lm0.model.exog_names` para respetar el orden. Usa `mod_lm0.predict(...)` y guarda las nuevas predicciones con otro nombre. No incluyas el resto de variables: el modelo simple no las conoce.

In [ ]:
# 3.2.1 — Predecir la resistencia de test con el modelo simple.
# Pista: construye la misma matriz con constante y Cement que en X_simple.
# Guarda las nuevas predicciones con otro nombre; conserva pr sin cambios.
# Tu código:

### 3.2.2. Comparar RMSE y concluir

Calcula el RMSE de las nuevas predicciones frente a `test['Strength']` y compáralo con `RMSE` del modelo final. Usa **las mismas 324 filas** y expresa ambos errores en **MPa**.

Redacta una conclusión con los dos valores, cuál es menor en este test y una limitación. ¿Por qué no basta comparar el $R^2$ de entrenamiento? Porque mide ajuste en los datos usados para estimar los coeficientes, no error sobre observaciones nuevas.

Si el modelo más complejo no gana, considera sobreajuste, incertidumbre y tamaño de la muestra; si gana, tampoco conviertas esa ventaja en una garantía para cada pieza. No modifiques el modelo final a partir de esta comparación.

In [ ]:
# 3.2.2 — Comparar el RMSE en test.
# Pista: calcula la raíz de la media de los errores cuadrados de las nuevas predicciones.
# Compara ese resultado con RMSE del modelo final y expresa ambos en MPa.
# No reajustes ni elijas modelos basándote en este conjunto de test.
# Tu código y conclusión:

## Cierre de la sesión

Hemos pasado de describir los datos a ajustar, seleccionar, diagnosticar y evaluar una regresión. El modelo final conserva **todas las 706 observaciones de entrenamiento**, siete variables con términos cuadráticos y `Strength` en MPa. Los ajustes transformado y reducido son exploraciones complementarias, no sustitutos elegidos a partir de test.

En las **324 observaciones de test**, el modelo obtiene aproximadamente **RMSE = 8,54 MPa**, **MAE = 6,46 MPa** y **$R^2_{test}=0{,}754$**. Son resultados de esta muestra reservada, no una garantía de precisión individual.

Para cerrar tu análisis, distingue siempre:

1. **Ajuste:** qué explican los coeficientes y el $R^2$ de entrenamiento.
2. **Selección:** qué simplificó el AIC y con qué respuesta y filas se comparó.
3. **Diagnóstico:** qué patrones mejoraron y cuáles siguen aconsejando cautela.
4. **Evaluación:** cuánto error se observó en test, con sus unidades y tolerancia práctica.

**No confundir:** asociación con causalidad, un residuo grande con un dato erróneo, un IC de la media con un IP individual, ni un buen ajuste de entrenamiento con buena predicción fuera de la muestra. Para revisar modelos después de usar este test harán falta datos nuevos de validación o un diseño de evaluación separado.

### Documentación de las herramientas

- [Regresión por mínimos cuadrados en statsmodels](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html).
- [Suavizado LOWESS](https://www.statsmodels.org/stable/generated/statsmodels.nonparametric.smoothers_lowess.lowess.html).
- [Transformación Box–Cox en SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.boxcox.html).